In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into kernel variables (full DataFrames kept for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Timezone for UTC+8 as required
tz = pytz.timezone('Asia/Shanghai')

# Helper to compute min/max timestamps in UTC+8 (if timestamp column exists)
def min_max_timestamps(df):
    if 'timestamp' in df.columns and len(df):
        times = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        return times.min(), times.max()
    return pd.NaT, pd.NaT

# Helper to get top N distinct cmdb_id values (alphabetical)
def top_cmdbs(df, n=20):
    if 'cmdb_id' in df.columns:
        vals = sorted(df['cmdb_id'].dropna().astype(str).unique())[:n]
        return pd.DataFrame({'cmdb_id': vals})
    return pd.DataFrame({'cmdb_id': []})

# Summaries for metric.csv
metric_count = metric_df.shape[0]
metric_min_ts, metric_max_ts = min_max_timestamps(metric_df)
metric_overview = pd.DataFrame({
    'row_count': [metric_count],
    'min_timestamp': [metric_min_ts],
    'max_timestamp': [metric_max_ts]
})
metric_top_cmdbs = top_cmdbs(metric_df, 20)

# metric-specific extras
if 'kpi_name' in metric_df.columns:
    metric_top_kpis = pd.DataFrame({'kpi_name': sorted(metric_df['kpi_name'].dropna().astype(str).unique())[:20]})
else:
    metric_top_kpis = pd.DataFrame({'kpi_name': []})

# component||kpi pairs with counts (top 20 by count)
if {'cmdb_id', 'kpi_name'}.issubset(metric_df.columns) and metric_count>0:
    pairs = (metric_df[['cmdb_id','kpi_name']]
             .fillna('')
             .astype(str)
             .assign(component_kpi=lambda d: d['cmdb_id'] + '||' + d['kpi_name']))
    metric_top_pairs = (pairs.groupby('component_kpi', as_index=False)
                        .size()
                        .rename(columns={'size':'count'})
                        .sort_values('count', ascending=False)
                        .head(20)
                        .reset_index(drop=True))
else:
    metric_top_pairs = pd.DataFrame(columns=['component_kpi','count'])

# Summaries for trace.csv
trace_count = trace_df.shape[0]
trace_min_ts, trace_max_ts = min_max_timestamps(trace_df)
trace_overview = pd.DataFrame({
    'row_count': [trace_count],
    'min_timestamp': [trace_min_ts],
    'max_timestamp': [trace_max_ts]
})
trace_top_cmdbs = top_cmdbs(trace_df, 20)
if 'trace_name' in trace_df.columns:
    trace_top_traces = pd.DataFrame({'trace_name': sorted(trace_df['trace_name'].dropna().astype(str).unique())[:20]})
else:
    trace_top_traces = pd.DataFrame({'trace_name': []})

# Summaries for log.csv
log_count = log_df.shape[0]
log_min_ts, log_max_ts = min_max_timestamps(log_df)
log_overview = pd.DataFrame({
    'row_count': [log_count],
    'min_timestamp': [log_min_ts],
    'max_timestamp': [log_max_ts]
})
log_top_cmdbs = top_cmdbs(log_df, 20)
if 'log_name' in log_df.columns:
    log_top_logs = pd.DataFrame({'log_name': sorted(log_df['log_name'].dropna().astype(str).unique())[:20]})
else:
    log_top_logs = pd.DataFrame({'log_name': []})

# Summaries for error_logs.csv
error_count = error_logs_df.shape[0]
error_min_ts, error_max_ts = min_max_timestamps(error_logs_df)
error_overview = pd.DataFrame({
    'row_count': [error_count],
    'min_timestamp': [error_min_ts],
    'max_timestamp': [error_max_ts]
})
error_top_cmdbs = top_cmdbs(error_logs_df, 20)

# top 10 distinct message samples shortened to 200 chars
if 'message' in error_logs_df.columns:
    uniq_msgs = error_logs_df['message'].dropna().astype(str).unique()[:10]
    msg_samples = [m if len(m)<=200 else m[:200] for m in uniq_msgs]
    error_message_samples = pd.DataFrame({'message_sample': msg_samples})
else:
    error_message_samples = pd.DataFrame({'message_sample': []})

# Return compact summaries (do not display full raw DataFrames)
metric_overview, metric_top_cmdbs, metric_top_kpis, metric_top_pairs, \
trace_overview, trace_top_cmdbs, trace_top_traces, \
log_overview, log_top_cmdbs, log_top_logs, \
error_overview, error_top_cmdbs, error_message_samples

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry (plain English):\n\n"
    "metric.csv: 144,252 rows spanning 2022-03-20 15:30 to 2022-03-20 15:59 (UTC+8). "
    "Top component examples include adservice, adservice-0/1/2, cartservice*, checkoutservice*, currencyservice*, emailservice*, and shippingservice. "
    "Top KPI categories include app.grpc.*, app.http.* and many container.node-5.* metrics. "
    "The top component||KPI pairs are dominated by container.node-5.* metrics (e.g., multiple adservice-0 and shippingservice entries), each appearing 30 times in the sample top-20.\n\n"
    "trace.csv: 36,720 rows spanning the same 15:30–15:59 window. "
    "Top component examples are pod-level names (adservice-0/1/2, cartservice-*, checkoutservice-*, etc.). "
    "Common trace names include per-pod traces like trace.from_<service>-<pod>.(duration_mean, duration_p95, error_rate, row_count).\n\n"
    "log.csv: 1,814 rows in the same time range. "
    "Observed log metric names are minimal: log.error_count and log.row_count.\n\n"
    "error_logs.csv: 1,988 rows from 2022-03-20 15:30:00 to 2022-03-20 15:59:59 (UTC+8). "
    "Top cmdb_id samples include frontend-0/1/2. "
    "A sample message (shortened) starts with: \"severity: warning, message: failed to retrieve...\".\n\n"
    "Overall: all four files loaded and retained in kernel variables. The metric and trace datasets are large (144k and 36k rows) covering the same half-hour window; logs and error logs are much smaller. "
    "Telemetry shows many container.node-5 metrics across multiple pods/services and per-pod traces for service-to-service calls. "
    "Data is ready for deeper analysis (anomaly detection, correlation of metrics/traces/logs) if you want to proceed."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   row_count             min_timestamp             max_timestamp
0     144252 2022-03-20 15:30:00+08:00 2022-03-20 15:59:00+08:00,                cmdb_id
0            adservice
1          adservice-0
2          adservice-1
3          adservice-2
4           adservice2
5         adservice2-0
6          cartservice
7        cartservice-0
8        cartservice-1
9        cartservice-2
10      cartservice2-0
11     checkoutservice
12   checkoutservice-0
13   checkoutservice-1
14   checkoutservice-2
15  checkoutservice2-0
16     currencyservice
17   currencyservice-0
18   currencyservice-1
19   currencyservice-2,                                              kpi_name
0                                      app.grpc.count
1                                        app.grpc.mrt
2                                         app.grpc.rr
3                                         app.grpc.sr
4                                      app.http.count
5                                        app.http.mrt
6                                         app.http.rr
7                                         app.http.sr
8          container.node-5.container_cpu_cfs_periods
9   container.node-5.container_cpu_cfs_throttled_p...
10  container.node-5.container_cpu_cfs_throttled_s...
11    container.node-5.container_cpu_load_average_10s
12      container.node-5.container_cpu_system_seconds
13       container.node-5.container_cpu_usage_seconds
14        container.node-5.container_cpu_user_seconds
15        container.node-5.container_file_descriptors
16     container.node-5.container_fs_inodes./dev/vda1
17  container.node-5.container_fs_inodes_free./dev...
18  container.node-5.container_fs_io_current./dev/...
19  container.node-5.container_fs_io_time_seconds....,                                         component_kpi  count
0                        shippingservice||app.grpc.sr     30
1   adservice-0||container.node-5.container_cpu_cf...     30
2   adservice-0||container.node-5.container_cpu_cf...     30
3   adservice-0||container.node-5.container_cpu_cf...     30
4   adservice-0||container.node-5.container_cpu_lo...     30
5   adservice-0||container.node-5.container_cpu_sy...     30
6   adservice-0||container.node-5.container_cpu_us...     30
7   adservice-0||container.node-5.container_cpu_us...     30
8   adservice-0||container.node-5.container_file_d...     30
9   adservice-0||container.node-5.container_fs_ino...     30
10  adservice-0||container.node-5.container_fs_ino...     30
11  adservice-0||container.node-5.container_fs_io_...     30
12  adservice-0||container.node-5.container_fs_io_...     30
13  adservice-0||container.node-5.container_fs_io_...     30
14  adservice-0||container.node-5.container_fs_lim...     30
15  adservice-0||container.node-5.container_fs_rea...     30
16  adservice-0||container.node-5.container_fs_rea...     30
17  adservice-0||container.node-5.container_fs_rea...     30
18  shippingservice2-0||container.node-5.container...     30
19  shippingservice2-0||container.node-5.container...     30,    row_count             min_timestamp             max_timestamp
0      36720 2022-03-20 15:30:00+08:00 2022-03-20 15:59:00+08:00,                cmdb_id
0          adservice-0
1          adservice-1
2          adservice-2
3         adservice2-0
4        cartservice-0
5        cartservice-1
6        cartservice-2
7       cartservice2-0
8    checkoutservice-0
9    checkoutservice-1
10   checkoutservice-2
11  checkoutservice2-0
12   currencyservice-0
13   currencyservice-1
14   currencyservice-2
15  currencyservice2-0
16      emailservice-0
17      emailservice-1
18      emailservice-2
19     emailservice2-0,                                      trace_name
0    trace.from_checkoutservice-0.duration_mean
1     trace.from_checkoutservice-0.duration_p95
2       trace.from_checkoutservice-0.error_rate
3        trace.from_checkoutservice-0.row_count
4    trace.from_checkoutservice-1.duration_mean
5     trace.from_checkoutservice-1.duration_p95
6       trace.from_checkoutservice-1.error_rate
7        trace.from_checkoutservice-1.row_count
8    trace.from_checkoutservice-2.duration_mean
9     trace.from_checkoutservice-2.duration_p95
10      trace.from_checkoutservice-2.error_rate
11       trace.from_checkoutservice-2.row_count
12  trace.from_checkoutservice2-0.duration_mean
13   trace.from_checkoutservice2-0.duration_p95
14     trace.from_checkoutservice2-0.error_rate
15      trace.from_checkoutservice2-0.row_count
16          trace.from_frontend-0.duration_mean
17           trace.from_frontend-0.duration_p95
18             trace.from_frontend-0.error_rate
19              trace.from_frontend-0.row_count,    row_count             min_timestamp             max_timestamp
0       1814 2022-03-20 15:30:00+08:00 2022-03-20 15:59:00+08:00,               cmdb_id
0         adservice-0
1         adservice-1
2         adservice-2
3       cartservice-0
4       cartservice-1
5       cartservice-2
6   checkoutservice-0
7   checkoutservice-1
8   checkoutservice-2
9   currencyservice-0
10  currencyservice-1
11  currencyservice-2
12     emailservice-0
13     emailservice-1
14     emailservice-2
15         frontend-0
16         frontend-1
17         frontend-2
18   paymentservice-0
19   paymentservice-1,           log_name
0  log.error_count
1    log.row_count,    row_count             min_timestamp             max_timestamp
0       1988 2022-03-20 15:30:00+08:00 2022-03-20 15:59:59+08:00,       cmdb_id
0  frontend-0
1  frontend-1
2  frontend-2,                                       message_sample
0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import numpy as np

# Reuse metric_df already in kernel; if not present, load it (safe-guard)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) across all available rows (global, no time filter)
p95_df = (metric_df
          .groupby(['cmdb_id', 'kpi_name'], dropna=False, as_index=False)['value']
          .quantile(0.95)
          .rename(columns={'value': 'global_P95'}))

# Keep full p95_df in memory for later
global_p95_df = p95_df

# 2) Filter metric rows to the incident window: 2022-03-20 15:30:00 (inclusive) to 2022-03-20 16:00:00 (exclusive) in UTC+8
start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# 3) For each pair within filtered window compute anomaly counts where value >= global_P95,
#    earliest anomaly timestamp in window, and max value in window.
# Merge global P95 into windowed rows
metric_window = metric_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomalies (value >= global_P95). If global_P95 missing (shouldn't), treat as no anomaly.
metric_window['is_anomaly'] = ((metric_window['value'] >= metric_window['global_P95']) & metric_window['global_P95'].notna()).astype(int)

# Aggregations per pair within window
agg = (metric_window
       .groupby(['cmdb_id','kpi_name'], as_index=False)
       .agg(total_points_in_window=('value','size'),
            anomaly_count=('is_anomaly','sum'),
            max_value_in_window=('value','max'),
            global_P95=('global_P95','first')))

# Earliest anomaly timestamp per pair (only where is_anomaly==1)
earliest_anom = (metric_window[metric_window['is_anomaly'] == 1]
                 .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
                 .min()
                 .rename(columns={'timestamp':'earliest_anomaly_ts'}))

# Merge earliest anomaly ts (may be missing for groups with zero anomalies)
agg = agg.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Convert earliest anomaly ts to timezone-aware readable string in UTC+8; keep NaT/NaN as None
def ts_to_shanghai_str(s):
    if pd.isna(s):
        return None
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_ts'].apply(ts_to_shanghai_str)
agg = agg.drop(columns=['earliest_anomaly_ts'])

# 4) Return compact table: top 20 pairs by anomaly_count (desc). If ties, sort by max_value_in_window desc.
top20 = (agg.sort_values(by=['anomaly_count','max_value_in_window'], ascending=[False, False])
          .reset_index(drop=True)
          .head(20))

# Keep full intermediate tables in kernel variables for later:
global_p95_df, metric_window, agg_full = global_p95_df, metric_window, agg

# Compact output: selected columns and rounding numeric columns for readability
top20_display = top20[['cmdb_id','kpi_name','total_points_in_window','anomaly_count',
                       'earliest_anomaly_timestamp','max_value_in_window','global_P95']].copy()
top20_display['max_value_in_window'] = top20_display['max_value_in_window'].round(6)
top20_display['global_P95'] = top20_display['global_P95'].round(6)

top20_display

```
Out[2]:
```
```python
# Summarize the analysis results in plain English using in-memory variables from prior steps
try:
    agg_full
    top20_display
except NameError:
    raise RuntimeError("Required intermediate variables (agg_full, top20_display) not found in kernel.")

total_pairs = agg_full.shape[0]
pairs_with_anomalies = int((agg_full['anomaly_count'] > 0).sum())
pct = pairs_with_anomalies / total_pairs * 100 if total_pairs else 0

# Prepare a short list of top 5 pairs for examples
examples = top20_display[['cmdb_id','kpi_name','anomaly_count']].head(5).to_dict(orient='records')
examples_txt = "; ".join([f"{r['cmdb_id']}||{r['kpi_name']} (anomalies={r['anomaly_count']})" for r in examples])

summary = (
    "Summary of metric anomaly scan (plain English):\n\n"
    f"- Computed a global P95 for each (cmdb_id, kpi_name) using the entire metric dataset.\n"
    "- Filtered metric rows to the incident window 2022-03-20 15:30:00 (inclusive) to 2022-03-20 16:00:00 (exclusive) in UTC+8.\n\n"
    f"- Total distinct (cmdb_id, kpi_name) pairs considered: {total_pairs}.\n"
    f"- Pairs with at least one point >= global P95 in the window: {pairs_with_anomalies} ({pct:.1f}%).\n\n"
    "Top findings (top 20 by anomaly count):\n"
    f"- Many top pairs have anomaly_count = total_points_in_window = 30, with earliest anomaly at 2022-03-20 15:30 (UTC+8),\n"
    "  and max_value equal to the global P95. Example top pairs: " + examples_txt + ".\n\n"
    "Interpretation:\n"
    "- The top-ranked 'anomalies' are dominated by largely constant or monotonic metrics (e.g., JVM/OS start times, container_start_time_seconds,\n"
    "  and system.disk.total). Because these metrics are static or represent timestamps/sizes, their values sit at or above the computed global P95\n"
    "  and thus appear as anomalies across the entire window. This pattern suggests the flagged items are not transient performance spikes,\n"
    "  but inherent constant/large metrics (start times, disk totals).\n\n"
    "Recommendation:\n"
    "- Focus further investigation on dynamic KPIs (CPU usage, memory usage, I/O rates, network latency, trace latency/error-rate) and correlate\n"
    "  with traces/logs for real transient anomalies. The current top results likely reflect non-actionable constants rather than runtime faults.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_P95
0                 adservice2                runtime.java_lang_Runtime_StartTime                      30             30   2022-03-20 15:30:00+0800         1.647523e+12  1.647523e+12
1                  adservice                runtime.java_lang_Runtime_StartTime                      30             30   2022-03-20 15:30:00+0800         1.647523e+12  1.647523e+12
2                     node-4                                  system.disk.total                      30             30   2022-03-20 15:30:00+0800         7.616300e+09  7.616300e+09
3                     node-1                                  system.disk.total                      30             30   2022-03-20 15:30:00+0800         5.867011e+09  5.867011e+09
4                     node-2                                  system.disk.total                      30             30   2022-03-20 15:30:00+0800         5.867011e+09  5.867011e+09
5                     node-3                                  system.disk.total                      30             30   2022-03-20 15:30:00+0800         5.867011e+09  5.867011e+09
6                 adservice2  runtime.java_lang_OperatingSystem_CommittedVir...                      30             30   2022-03-20 15:30:00+0800         3.703128e+09  3.703128e+09
7                     node-6                                  system.disk.total                      30             30   2022-03-20 15:30:00+0800         3.617924e+09  3.617924e+09
8                  adservice  runtime.java_lang_OperatingSystem_CommittedVir...                      30             30   2022-03-20 15:30:00+0800         3.589956e+09  3.589956e+09
9             cartservice2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647735e+09  1.647735e+09
10             cartservice-0      container.node-5.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647726e+09  1.647726e+09
11             cartservice-1      container.node-5.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647725e+09  1.647725e+09
12             cartservice-2      container.node-5.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647723e+09  1.647723e+09
13  productcatalogservice2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09
14        currencyservice2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09
15         paymentservice2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09
16  recommendationservice2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09
17              adservice2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09
18               frontend2-0      container.node-6.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09
19   productcatalogservice-2      container.node-5.container_start_time_seconds                      30             30   2022-03-20 15:30:00+0800         1.647523e+09  1.647523e+09```
```

In [ ]:
# In[3]:

import pandas as pd
import re
import pytz

# Reuse metric_df if present, otherwise load (safe-guard)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) Keywords (case-insensitive) to identify dynamic KPI candidates
keywords = [
    'cpu', 'memory', 'mem', 'load', 'io', 'fs_io', 'io_time',
    'network', 'net', 'packet', 'latency',
    'app.http', 'app.grpc', 'http.mrt', 'grpc.mrt', 'mrt', 'duration'
]
# Build regex pattern to search any keyword as substring (case-insensitive)
# Sort by length descending to avoid partial collisions (not strictly necessary)
escaped = [re.escape(k) for k in sorted(keywords, key=len, reverse=True)]
pattern = '(' + '|'.join(escaped) + ')'

# 1) Identify distinct kpi_name values matching any keyword
kpi_series = metric_df['kpi_name'].astype(str)
mask = kpi_series.str.contains(pattern, case=False, na=False, regex=True)
matched_kpi_names = pd.Series(kpi_series[mask].unique(), name='kpi_name').sort_values().reset_index(drop=True)

# 2) For each matching kpi_name, return total row count across full dataset (no time filter). Top 50 by count.
kpi_counts = (metric_df[mask]
              .groupby('kpi_name', dropna=False, as_index=False)
              .size()
              .rename(columns={'size':'total_count'})
              .sort_values('total_count', ascending=False)
              .reset_index(drop=True))

top_kpi_counts = kpi_counts.head(50)

# 3) For the top 20 kpi_names, compute top 10 cmdb_id values by row count for that kpi_name
top20_kpis = top_kpi_counts['kpi_name'].head(20).tolist()

# Group by kpi_name and cmdb_id to get counts, then filter to top20_kpis and pick top10 per kpi
grouped = (metric_df[metric_df['kpi_name'].isin(top20_kpis)]
           .groupby(['kpi_name','cmdb_id'], dropna=False, as_index=False)
           .size()
           .rename(columns={'size':'count'}))

# For each kpi_name, get top 10 cmdb_id by count
top20_cmdb_counts = (grouped
                     .sort_values(['kpi_name','count'], ascending=[True, False])
                     .groupby('kpi_name', as_index=False)
                     .head(10)
                     .reset_index(drop=True))

# Keep full intermediate results in kernel variables
matched_kpi_names_full = matched_kpi_names
kpi_counts_full = kpi_counts
grouped_full = grouped

# Compact outputs requested:
top_kpi_counts, top20_cmdb_counts

```
Out[3]:
```
```python
# Create a compact plain-English summary of the discovered dynamic KPI candidates
try:
    top_kpi_counts
    top20_cmdb_counts
except NameError:
    raise RuntimeError("Expected intermediate DataFrames (top_kpi_counts, top20_cmdb_counts) not found.")

# Basic stats
num_matched_kpis = int(top_kpi_counts.shape[0])

# Top 10 KPI names with counts (compact)
top10_kpis = top_kpi_counts.head(10)[['kpi_name','total_count']].values.tolist()
top10_lines = [f"{i+1}. {k} (rows={c})" for i,(k,c) in enumerate(top10_kpis)]

# For top 5 KPI names, list top cmdb_id by count (up to 10 each)
top5_kpis = top_kpi_counts['kpi_name'].head(5).tolist()
top5_cmdb_lines = []
for k in top5_kpis:
    rows = top20_cmdb_counts[top20_cmdb_counts['kpi_name']==k][['cmdb_id','count']].head(10).values.tolist()
    rows_str = ", ".join([f"{cmdb}({cnt})" for cmdb,cnt in rows])
    top5_cmdb_lines.append(f"- {k}: {rows_str}")

summary = (
    f"Dynamic KPI candidate discovery (plain English):\n\n"
    f"- Identified {num_matched_kpis} distinct kpi_name values matching keywords like cpu/mem/io/net/http/grpc/mrt/duration.\n\n"
    f"- Top 10 KPI names by total row count (full dataset):\n" +
    "\n".join(top10_lines) + "\n\n"
    f"- For the top 5 KPIs, top cmdb_id contributors (cmdb_id(count), up to 10 each):\n" +
    "\n".join(top5_cmdb_lines) + "\n\n"
    "Notes:\n"
    "- Many top KPIs are container.node-5 or node-6 metrics (network, transmit/receive MB, packet/error counts, cpu system seconds, memory RSS, fs I/O time, etc.).\n"
    "- These KPIs are good dynamic candidates for further anomaly detection and correlation with traces/logs.\n"
    "- Next step: run anomaly detection on these dynamic KPIs (e.g., CPU, memory, network, I/O, app.http/app.grpc latency) within the incident window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  total_count
0   container.node-5.container_network_receive_MB....         1050
1   container.node-5.container_network_transmit_er...         1050
2   container.node-5.container_network_transmit_pa...         1050
3   container.node-5.container_network_transmit_pa...         1050
4   container.node-5.container_network_transmit_MB...         1050
..                                                ...          ...
45              container.node-6.container_memory_rss          210
46    container.node-6.container_cpu_load_average_10s          210
47  container.node-6.container_fs_io_time_weighted...          210
48            container.node-6.container_memory_cache          210
49  container.node-6.container_cpu_cfs_throttled_s...          210

[50 rows x 2 columns],                                              kpi_name             cmdb_id  count
0       container.node-5.container_cpu_system_seconds         adservice-0     30
1       container.node-5.container_cpu_system_seconds         adservice-1     30
2       container.node-5.container_cpu_system_seconds         adservice-2     30
3       container.node-5.container_cpu_system_seconds       cartservice-0     30
4       container.node-5.container_cpu_system_seconds       cartservice-1     30
..                                                ...                 ...    ...
195  container.node-5.container_tasks_state.iowaiting       cartservice-2     30
196  container.node-5.container_tasks_state.iowaiting   checkoutservice-0     30
197  container.node-5.container_tasks_state.iowaiting   checkoutservice-1     30
198  container.node-5.container_tasks_state.iowaiting   checkoutservice-2     30
199  container.node-5.container_tasks_state.iowaiting  checkoutservice2-0     30

[200 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing in-memory variables where available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# top_kpi_counts should exist from previous step; else recompute dynamic KPI counts
try:
    top_kpi_counts
except NameError:
    # fallback: identify dynamic KPIs as in previous step (simple safe fallback)
    keywords = ['cpu','memory','mem','load','io','fs_io','io_time','network','net','packet','latency',
                'app.http','app.grpc','http.mrt','grpc.mrt','mrt','duration']
    pattern = '(' + '|'.join([pd.re.escape(k) for k in sorted(keywords, key=len, reverse=True)]) + ')'
    mask = metric_df['kpi_name'].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    top_kpi_counts = (metric_df[mask].groupby('kpi_name', as_index=False).size().rename(columns={'size':'total_count'})
                      .sort_values('total_count', ascending=False).reset_index(drop=True))

# 1) Take top 20 dynamic kpis by total_count
top20_kpis = top_kpi_counts['kpi_name'].head(20).tolist()

# 2) Compute global P95 for every (cmdb_id, kpi_name) pair for those kpis
# Prefer reusing previously computed global_p95_df if present (was computed earlier); otherwise compute now.
try:
    global_p95_df
except NameError:
    global_p95_df = (metric_df
                     .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
                     .quantile(0.95)
                     .rename(columns={'value':'global_P95'}))

p95_top20_df = global_p95_df[global_p95_df['kpi_name'].isin(top20_kpis)].copy()

# 3) Filter metric rows to the incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# 4) Restrict to top-20 kpis and merge global P95
metric_window_top20 = metric_window[metric_window['kpi_name'].isin(top20_kpis)].merge(
    p95_top20_df, on=['cmdb_id','kpi_name'], how='left'
)

# Flag anomalies where value >= global_P95 (and global_P95 not null)
metric_window_top20['is_anomaly'] = ((metric_window_top20['value'] >= metric_window_top20['global_P95']) &
                                     metric_window_top20['global_P95'].notna()).astype(int)

# Aggregations per (cmdb_id, kpi_name)
agg_top20 = (metric_window_top20
             .groupby(['cmdb_id','kpi_name'], as_index=False)
             .agg(total_points_in_window=('value','size'),
                  anomaly_count=('is_anomaly','sum'),
                  max_value_in_window=('value','max'),
                  global_P95=('global_P95','first')))

# Earliest anomaly timestamp (numeric) per pair
earliest_anom = (metric_window_top20[metric_window_top20['is_anomaly'] == 1]
                 .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
                 .min()
                 .rename(columns={'timestamp':'earliest_anom_ts'}))

agg_top20 = agg_top20.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# For sorting, keep numeric earliest_anom_ts; then convert to UTC+8 readable string for display
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# Sort by anomaly_count desc, earliest_anom_ts asc (nulls last)
agg_top20_sorted = agg_top20.sort_values(by=['anomaly_count','earliest_anom_ts'],
                                         ascending=[False, True],
                                         na_position='last').reset_index(drop=True)

# Prepare compact display: convert earliest timestamp to string, round numeric columns
agg_top20_sorted['earliest_anomaly_timestamp'] = agg_top20_sorted['earliest_anom_ts'].apply(fmt_ts)
display_cols = ['cmdb_id','kpi_name','total_points_in_window','anomaly_count',
                'earliest_anomaly_timestamp','max_value_in_window','global_P95']
top20_display = agg_top20_sorted[display_cols].copy()
top20_display['max_value_in_window'] = top20_display['max_value_in_window'].round(6)
top20_display['global_P95'] = top20_display['global_P95'].round(6)

# Keep full intermediate tables cached for later use
p95_top20_df_full = p95_top20_df
metric_window_top20_full = metric_window_top20
agg_top20_full = agg_top20

# Return compact top-20 table
top20_display.head(20)

```
Out[4]:
```
```python
summary = (
    "Summary of dynamic-KPI scan (plain English):\n\n"
    "- I evaluated the top-20 dynamic KPI names (by total rows) for all (cmdb_id, kpi_name) pairs and computed global P95 per pair, "
    "then inspected the incident window 2022-03-20 15:30:00–16:00:00 (UTC+8).\n\n"
    "- Key observation: many top pairs have total_points_in_window = 30 and anomaly_count = 30, with the earliest anomaly at 2022-03-20 15:30 (UTC+8). "
    "Their max_value_in_window equals the computed global_P95. Example top pairs include multiple adservice pods (adservice-0, adservice-1) for container.node-5 metrics "
    "such as fs_io_time, fs_io_current, memory_cache, memory_mapped_file, network error/packet counters, and spec_memory_swap_limit.\n\n"
    "- Interpretation: these 'anomalies' are mostly due to constant/static or monotonically large metrics (zeros, start-times, cached memory sizes, counters), "
    "so using a global P95 threshold flags every sample in the window. This pattern indicates non-transient, non-actionable flags rather than runtime spikes.\n\n"
    "- Recommendation / next steps:\n"
    "  1) Focus anomaly detection on truly dynamic metrics (CPU utilization, memory RSS, network MB/s, I/O rates, latency metrics like app.http/app.grpc/mrt or duration) and/or\n"
    "     exclude static/time-invariant KPIs (start times, total disk size, constant counters) from threshold-based detection.\n"
    "  2) For thresholding, consider alternative methods (rate of change, z-score, baseline over time-of-day) or require deviation from recent baseline rather than global P95 for static counters.\n"
    "  3) Correlate dynamic KPI anomalies with traces/logs for actionable investigation.\n\n"
    "If you want, I can now (a) filter these top-20 KPIs to only those with non-trivial variance, or (b) run anomaly detection on a curated set of CPU/memory/network/latency KPIs."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_P95
0   adservice-0  container.node-5.container_fs_io_current./dev/...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
1   adservice-0  container.node-5.container_fs_io_time_seconds....                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
2   adservice-0  container.node-5.container_fs_io_time_weighted...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
3   adservice-0            container.node-5.container_memory_cache                      30             30   2022-03-20 15:30:00+0800           57778176.0  57778176.0
4   adservice-0          container.node-5.container_memory_failcnt                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
5   adservice-0  container.node-5.container_memory_failures.hie...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
6   adservice-0      container.node-5.container_memory_mapped_file                      30             30   2022-03-20 15:30:00+0800           16564224.0  16564224.0
7   adservice-0  container.node-5.container_network_receive_err...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
8   adservice-0  container.node-5.container_network_receive_pac...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
9   adservice-0  container.node-5.container_network_transmit_er...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
10  adservice-0  container.node-5.container_network_transmit_pa...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
11  adservice-0  container.node-5.container_spec_memory_swap_li...                      30             30   2022-03-20 15:30:00+0800                600.0       600.0
12  adservice-0   container.node-5.container_tasks_state.iowaiting                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
13  adservice-1  container.node-5.container_fs_io_current./dev/...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
14  adservice-1  container.node-5.container_fs_io_time_seconds....                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
15  adservice-1  container.node-5.container_fs_io_time_weighted...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
16  adservice-1            container.node-5.container_memory_cache                      30             30   2022-03-20 15:30:00+0800            3133440.0   3133440.0
17  adservice-1          container.node-5.container_memory_failcnt                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
18  adservice-1  container.node-5.container_memory_failures.hie...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
19  adservice-1      container.node-5.container_memory_mapped_file                      30             30   2022-03-20 15:30:00+0800              53248.0     53248.0```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse in-memory trace_df if available; otherwise load (safe-guard)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, trace_name) using all available rows (no time filter)
global_trace_p95_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False, as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter trace rows to the incident window: 2022-03-20 15:30:00 (inclusive) to 2022-03-20 16:00:00 (exclusive) UTC+8
start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())

trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)].copy()

# 3) Merge global P95 into the windowed traces and flag anomalies (value >= global_P95)
trace_window_top = trace_window.merge(global_trace_p95_df, on=['cmdb_id', 'trace_name'], how='left')
trace_window_top['is_anomaly'] = ((trace_window_top['value'] >= trace_window_top['global_P95']) &
                                  trace_window_top['global_P95'].notna()).astype(int)

# 4) Aggregate per (cmdb_id, trace_name) within the window
agg_trace = (
    trace_window_top
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(total_points_in_window=('value','size'),
         anomaly_count=('is_anomaly','sum'),
         max_value_in_window=('value','max'),
         global_P95=('global_P95','first'))
)

# Earliest anomaly timestamp per group (if any)
earliest_anom = (
    trace_window_top[trace_window_top['is_anomaly'] == 1]
    .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anom_ts'})
)

agg_trace_full = agg_trace.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Helper to format timestamp to UTC+8 readable string
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

agg_trace_full['earliest_anomaly_timestamp'] = agg_trace_full['earliest_anom_ts'].apply(fmt_ts)

# 5) Prepare compact top-20 table sorted by anomaly_count desc, then earliest_anomaly_timestamp asc
agg_trace_sorted = agg_trace_full.sort_values(
    by=['anomaly_count','earliest_anom_ts'],
    ascending=[False, True],
    na_position='last'
).reset_index(drop=True)

display_cols = ['cmdb_id','trace_name','total_points_in_window','anomaly_count',
                'earliest_anomaly_timestamp','max_value_in_window','global_P95']

top20_trace_anomalies = agg_trace_sorted[display_cols].copy()
top20_trace_anomalies['max_value_in_window'] = top20_trace_anomalies['max_value_in_window'].round(6)
top20_trace_anomalies['global_P95'] = top20_trace_anomalies['global_P95'].round(6)

# Keep full intermediate tables cached: global_trace_p95_df, trace_window, trace_window_top, agg_trace_full
global_trace_p95_df, trace_window, trace_window_top, agg_trace_full, top20_trace_anomalies.head(20)

```
Out[5]:
```
```python
summary = (
    "Trace-level anomaly scan (plain English):\n\n"
    "- I computed global P95 per (cmdb_id, trace_name) across all trace data, then evaluated the incident window 2022-03-20 15:30–16:00 (UTC+8).\n\n"
    "- High-level counts: 1,232 distinct (cmdb_id, trace_name) pairs were assessed; 36,720 trace rows fall inside the incident window.\n\n"
    "- Key finding: many trace.error_rate series have global_P95 == 0.0 and value == 0.0, so using the rule \"value >= P95\" marks every sample as an \"anomaly\" (anomaly_count = 30, earliest = 15:30)."
    " These are non-actionable because they reflect constant zero error rates rather than transient failures.\n\n"
    "- More actionable signals to investigate:\n"
    "  * Per-pod duration metrics (duration_mean, duration_p95) show smaller, meaningful anomaly counts (not full-window flags). Example:\n"
    "    - adservice-0, trace.from_frontend-0.duration_mean: anomaly_count = 2, earliest anomaly = 2022-03-20 15:48 (UTC+8), max value > global_P95.\n"
    "    - adservice-0, trace.from_frontend-0.duration_p95: anomaly_count = 2, earliest = 2022-03-20 15:32 (UTC+8).\n"
    "  * Row_count changes also appear: e.g., adservice-0 trace.from_frontend-0.row_count has anomaly_count = 2 (earliest 15:46), and shippingservice2-0 trace.from_checkoutservice2-0.row_count shows anomaly_count = 9 (earliest 15:30), indicating traffic/load shifts.\n\n"
    "- Interpretation: error_rate zeros dominate the top-ranked \"anomalies\" due to the chosen thresholding rule; they are false positives in the sense of not representing runtime errors. The duration and row_count traces, with non-trivial max values and scattered anomaly timestamps, are better candidates to locate real incidents and timing.\n\n"
    "- Recommended next steps:\n"
    "  1) Ignore or treat specially trace.error_rate series where global_P95 == 0 (or require strict > P95 instead of >=) to avoid trivial flags.\n"
    "  2) Prioritize traces with 'duration' and 'row_count' where anomaly_count > 0 and max_value > global_P95 for root-cause correlation across services/pods.\n"
    "  3) Correlate the earliest anomaly timestamps from those duration/row_count traces with metric anomalies (CPU/memory/io) and logs to pinpoint affected services and timing.\n\n"
    "If you want, I can now produce a filtered list of trace anomalies excluding zero-error-rate series and show the top trace pairs (by anomaly_count and nonzero max) for follow-up investigation."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                               trace_name  global_P95
0            adservice-0      trace.from_frontend-0.duration_mean    0.000015
1            adservice-0       trace.from_frontend-0.duration_p95    0.000021
2            adservice-0         trace.from_frontend-0.error_rate    0.000000
3            adservice-0          trace.from_frontend-0.row_count   27.550000
4            adservice-0      trace.from_frontend-1.duration_mean    0.000014
...                  ...                                      ...         ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count    8.000000
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean    0.000078
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95    0.000147
1230  shippingservice2-0        trace.from_frontend2-0.error_rate    0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count   18.000000

[1232 rows x 3 columns],         timestamp             cmdb_id                               trace_name      value
0      1647761400         adservice-0      trace.from_frontend-0.duration_mean   0.000013
1      1647761400         adservice-0       trace.from_frontend-0.duration_p95   0.000018
2      1647761400         adservice-0         trace.from_frontend-0.error_rate   0.000000
3      1647761400         adservice-0          trace.from_frontend-0.row_count  22.000000
4      1647761400         adservice-0      trace.from_frontend-1.duration_mean   0.000012
...           ...                 ...                                      ...        ...
36715  1647763140  shippingservice2-0  trace.from_checkoutservice2-0.row_count   4.000000
36716  1647763140  shippingservice2-0     trace.from_frontend2-0.duration_mean   0.000061
36717  1647763140  shippingservice2-0      trace.from_frontend2-0.duration_p95   0.000068
36718  1647763140  shippingservice2-0        trace.from_frontend2-0.error_rate   0.000000
36719  1647763140  shippingservice2-0         trace.from_frontend2-0.row_count  13.000000

[36720 rows x 4 columns],         timestamp             cmdb_id                               trace_name      value  global_P95  is_anomaly
0      1647761400         adservice-0      trace.from_frontend-0.duration_mean   0.000013    0.000015           0
1      1647761400         adservice-0       trace.from_frontend-0.duration_p95   0.000018    0.000021           0
2      1647761400         adservice-0         trace.from_frontend-0.error_rate   0.000000    0.000000           1
3      1647761400         adservice-0          trace.from_frontend-0.row_count  22.000000   27.550000           0
4      1647761400         adservice-0      trace.from_frontend-1.duration_mean   0.000012    0.000014           0
...           ...                 ...                                      ...        ...         ...         ...
36715  1647763140  shippingservice2-0  trace.from_checkoutservice2-0.row_count   4.000000    8.000000           0
36716  1647763140  shippingservice2-0     trace.from_frontend2-0.duration_mean   0.000061    0.000078           0
36717  1647763140  shippingservice2-0      trace.from_frontend2-0.duration_p95   0.000068    0.000147           0
36718  1647763140  shippingservice2-0        trace.from_frontend2-0.error_rate   0.000000    0.000000           1
36719  1647763140  shippingservice2-0         trace.from_frontend2-0.row_count  13.000000   18.000000           0

[36720 rows x 6 columns],                  cmdb_id                               trace_name  total_points_in_window  anomaly_count  max_value_in_window  global_P95  earliest_anom_ts earliest_anomaly_timestamp
0            adservice-0      trace.from_frontend-0.duration_mean                      30              2             0.000023    0.000015        1647762480   2022-03-20 15:48:00+0800
1            adservice-0       trace.from_frontend-0.duration_p95                      30              2             0.000026    0.000021        1647761520   2022-03-20 15:32:00+0800
2            adservice-0         trace.from_frontend-0.error_rate                      30             30             0.000000    0.000000        1647761400   2022-03-20 15:30:00+0800
3            adservice-0          trace.from_frontend-0.row_count                      30              2            30.000000   27.550000        1647762360   2022-03-20 15:46:00+0800
4            adservice-0      trace.from_frontend-1.duration_mean                      30              3             0.000019    0.000014        1647762840   2022-03-20 15:54:00+0800
...                  ...                                      ...                     ...            ...                  ...         ...               ...                        ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count                      30              9             8.000000    8.000000        1647761400   2022-03-20 15:30:00+0800
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean                      30              2             0.000083    0.000078        1647762480   2022-03-20 15:48:00+0800
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95                      30              2             0.000174    0.000147        1647762480   2022-03-20 15:48:00+0800
1230  shippingservice2-0        trace.from_frontend2-0.error_rate                      30             30             0.000000    0.000000        1647761400   2022-03-20 15:30:00+0800
1231  shippingservice2-0         trace.from_frontend2-0.row_count                      30              4            19.000000   18.000000        1647762000   2022-03-20 15:40:00+0800

[1232 rows x 8 columns],           cmdb_id                                  trace_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_P95
0     adservice-0            trace.from_frontend-0.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
1     adservice-0            trace.from_frontend-1.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
2     adservice-0            trace.from_frontend-2.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
3     adservice-1            trace.from_frontend-0.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
4     adservice-1            trace.from_frontend-1.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
5     adservice-1            trace.from_frontend-2.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
6     adservice-2            trace.from_frontend-0.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
7     adservice-2            trace.from_frontend-1.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
8     adservice-2            trace.from_frontend-2.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
9    adservice2-0           trace.from_frontend2-0.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
10  cartservice-0  trace.from_checkoutservice-0.duration_mean                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
11  cartservice-0   trace.from_checkoutservice-0.duration_p95                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
12  cartservice-0     trace.from_checkoutservice-0.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
13  cartservice-0  trace.from_checkoutservice-1.duration_mean                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
14  cartservice-0   trace.from_checkoutservice-1.duration_p95                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
15  cartservice-0     trace.from_checkoutservice-1.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
16  cartservice-0  trace.from_checkoutservice-2.duration_mean                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
17  cartservice-0   trace.from_checkoutservice-2.duration_p95                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
18  cartservice-0     trace.from_checkoutservice-2.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
19  cartservice-0            trace.from_frontend-0.error_rate                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re
import pytz

# Ensure required in-memory variables exist
required_vars = ['trace_df', 'trace_window', 'trace_window_top', 'agg_trace_full', 'global_trace_p95_df']
for v in required_vars:
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel. Please run prior trace aggregation steps.")

tz = pytz.timezone('Asia/Shanghai')

# Work from the aggregated trace results
agg = agg_trace_full.copy()

# Normalize numeric columns
agg['max_value_in_window'] = pd.to_numeric(agg['max_value_in_window'], errors='coerce').fillna(-np.inf)
agg['global_P95'] = pd.to_numeric(agg['global_P95'], errors='coerce').fillna(-np.inf)
agg['anomaly_count'] = pd.to_numeric(agg['anomaly_count'], errors='coerce').fillna(0).astype(int)

# 1) Select pairs where anomaly_count > 0 AND (max_value_in_window > global_P95 OR global_P95 > 0)
mask_dynamic = (agg['anomaly_count'] > 0) & ((agg['max_value_in_window'] > agg['global_P95']) | (agg['global_P95'] > 0))

# 2) Restrict to trace_name containing specific keywords
trace_keywords = ['duration', 'duration_p95', 'duration_mean', 'row_count', 'error_rate']
pattern = '(' + '|'.join([re.escape(k) for k in trace_keywords]) + ')'
mask_name = agg['trace_name'].astype(str).str.contains(pattern, case=False, na=False, regex=True)

selected = agg[mask_dynamic & mask_name].copy()

# Ensure earliest anomaly timestamp numeric column exists; compute from trace_window_top if missing
if 'earliest_anom_ts' not in selected.columns or selected['earliest_anom_ts'].isnull().all():
    earliest = (trace_window_top[trace_window_top['is_anomaly']==1]
                .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
                .min()
                .rename(columns={'timestamp':'earliest_anom_ts'}))
    selected = selected.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# Format earliest_anomaly_timestamp to UTC+8 readable string
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

selected['earliest_anomaly_timestamp'] = selected['earliest_anom_ts'].apply(fmt_ts)

# Prepare compact output columns
output_cols = ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_P95']
compact_result = selected[output_cols].copy()

# Sort chronologically by earliest_anom_ts ascending, nulls last, limit to top 50
compact_result = compact_result.assign(_earliest_sort=selected['earliest_anom_ts'])
compact_result = compact_result.sort_values(by=['_earliest_sort'], ascending=True, na_position='last').drop(columns=['_earliest_sort']).reset_index(drop=True)
compact_result_top50 = compact_result.head(50)

# Keep full intermediate table cached
selected_trace_anomalies_full = selected

# Return compact top-50 table
compact_result_top50

```
Out[6]:
```
Summary of chronological, non-trivial trace anomalies (plain English):

- Timeline: Many actionable trace anomalies start at 2022-03-20 15:30:00 (UTC+8). The compact list (top 50) shows the earliest anomaly timestamps clustered at 15:30 with subsequent anomalies scattered across the 15:30–16:00 window. This indicates the incident onset around 15:30.

- Key types of trace signals observed:
  - row_count changes (traffic/load shifts) — e.g., shippingservice2-0 trace.from_checkoutservice2-0.row_count (anomaly_count=9, earliest 15:30), many checkoutservice → other-service row_count entries.
  - duration/duration_p95/duration_mean spikes — e.g., cartservice pods show many duration anomalies (cartservice-0 trace.self.duration_mean anomaly_count=30 at 15:30; cartservice-1 trace.from_checkoutservice-2.duration_mean/p95 anomaly_count=30 at 15:30).
  - Several services/pods repeatedly flagged (high anomaly_count): cartservice (multiple pods), shippingservice and shippingservice2, checkoutservice, productcatalogservice, adservice and others appear in the top list.

- What this implies:
  - The incident likely affected request latency and traffic patterns starting at ~15:30, impacting multiple services/pods (cartservice, shippingservice, checkoutservice, productcatalogservice, etc.).
  - row_count anomalies point to changes in traffic distribution or request rates; duration_p95/duration_mean anomalies point to latency degradation for affected call paths. These are better for root-cause than trivial zero-error_rate flags.

- Recommended next steps:
  1. Correlate these earliest anomaly timestamps (~15:30 and thereafter) with metric anomalies for the same pods/services (CPU, memory, network, I/O) and with logs around those timestamps.
  2. Prioritize traces with elevated duration_p95 and increased row_count as likely symptom sources to investigate (latency + load correlation).
  3. Inspect pod-level metrics (CPU/memory spikes, I/O or network issues) for the pods listed above (cartservice-*, shippingservice-*, checkoutservice-*, productcatalogservice-*, adservice-*) at ~15:30 to identify resource bottlenecks or degradations.
  4. If desired, I can now correlate these trace timepoints with the in-memory metric anomalies and error logs for those specific pods/services.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_P95
0        shippingservice2-0     trace.from_checkoutservice2-0.row_count              9   2022-03-20 15:30:00+0800             8.000000    8.000000
1             cartservice-0                    trace.self.duration_mean             30   2022-03-20 15:30:00+0800             0.000017    0.000000
2             cartservice-1  trace.from_checkoutservice-2.duration_mean             30   2022-03-20 15:30:00+0800             0.000750    0.000000
3             cartservice-1   trace.from_checkoutservice-2.duration_p95             30   2022-03-20 15:30:00+0800             0.002550    0.000000
4         shippingservice-2      trace.from_checkoutservice-1.row_count             11   2022-03-20 15:30:00+0800             4.000000    4.000000
..                      ...                                         ...            ...                        ...                  ...         ...
45        checkoutservice-1        trace.to_currencyservice-2.row_count              7   2022-03-20 15:30:00+0800             7.000000    6.000000
46        checkoutservice-1        trace.to_currencyservice-1.row_count              5   2022-03-20 15:30:00+0800             6.000000    6.000000
47        checkoutservice-1        trace.to_currencyservice-0.row_count              6   2022-03-20 15:30:00+0800             6.000000    6.000000
48  productcatalogservice-1  trace.from_checkoutservice-2.duration_mean              2   2022-03-20 15:30:00+0800             0.004991    0.004844
49       checkoutservice2-0          trace.to_emailservice2-0.row_count              9   2022-03-20 15:30:00+0800             4.000000    4.000000

[50 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import re
import numpy as np

# Reuse cached variables
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Use selected_trace_anomalies_full if present, else fall back to compact_result_top50 or agg_trace_full
if 'selected_trace_anomalies_full' in globals():
    trace_anom_df = selected_trace_anomalies_full.copy()
elif 'compact_result_top50' in globals():
    trace_anom_df = compact_result_top50.copy()
elif 'agg_trace_full' in globals():
    trace_anom_df = agg_trace_full.copy()
else:
    raise RuntimeError("Trace-anomaly aggregated table not found in kernel. Please run prior trace anomaly steps.")

# Timezone and window
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())

# 1) Top 10 cmdb_id by frequency in trace anomalies (robust reset)
vc = trace_anom_df['cmdb_id'].value_counts().head(10)
top10_cmdbs = vc.reset_index()
top10_cmdbs.columns = ['cmdb_id','freq']

# 2) For each top-10 cmdb_id, find metric kpi_name values matching keywords (case-insensitive), up to 20 per cmdb_id by row count
keywords = ['cpu','memory','memory_rss','mem','network','net','fs_io','io_time','io','disk','load','app.http','app.grpc','duration']
pattern = '(' + '|'.join([re.escape(k) for k in sorted(set(keywords), key=len, reverse=True)]) + ')'

top_cmdb_list = top10_cmdbs['cmdb_id'].tolist()
metric_candidates = metric_df[metric_df['cmdb_id'].isin(top_cmdb_list)].copy()
metric_candidates['kpi_name'] = metric_candidates['kpi_name'].astype(str)

mask_kpi = metric_candidates['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
metric_candidates = metric_candidates[mask_kpi]

# Count per (cmdb_id, kpi_name)
kpi_counts = (metric_candidates
              .groupby(['cmdb_id','kpi_name'], as_index=False)
              .size()
              .rename(columns={'size':'total_count'})
              .sort_values(['cmdb_id','total_count'], ascending=[True, False])
             )

# Keep up to 20 kpis per cmdb_id
top_kpis_per_cmdb = kpi_counts.groupby('cmdb_id', as_index=False).head(20).reset_index(drop=True)

# 3) Compute global P95 for each (cmdb_id, kpi_name) pair (use cached global_p95_df if exists)
pairs_to_compute = top_kpis_per_cmdb[['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)
if 'global_p95_df' in globals():
    p95_all = global_p95_df.copy()
else:
    p95_all = (metric_df
               .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
               .quantile(0.95)
               .rename(columns={'value':'global_P95'}))

p95_pairs = pairs_to_compute.merge(p95_all, on=['cmdb_id','kpi_name'], how='left')

# Compute any missing P95s explicitly
missing_mask = p95_pairs['global_P95'].isna()
if missing_mask.any():
    missing_pairs = p95_pairs[missing_mask][['cmdb_id','kpi_name']]
    missing_p95 = (metric_df.merge(missing_pairs, on=['cmdb_id','kpi_name'], how='inner')
                   .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
                   .quantile(0.95)
                   .rename(columns={'value':'global_P95'}))
    p95_pairs = p95_pairs.drop(columns=['global_P95']).merge(missing_p95, on=['cmdb_id','kpi_name'], how='left')

p95_pairs_full = p95_pairs.copy()

# 4) Filter metric rows to incident window and compute aggregates for candidate pairs
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# Join windowed rows with p95_pairs to restrict to candidate pairs
metric_window_candidates = metric_window.merge(p95_pairs[['cmdb_id','kpi_name','global_P95']], on=['cmdb_id','kpi_name'], how='inner')

# Flag anomalies (value >= global_P95 and global_P95 not null)
metric_window_candidates['is_anomaly'] = ((metric_window_candidates['value'] >= metric_window_candidates['global_P95']) &
                                          metric_window_candidates['global_P95'].notna()).astype(int)

# Aggregate per pair
agg_pairs = (metric_window_candidates
             .groupby(['cmdb_id','kpi_name'], as_index=False)
             .agg(total_points_in_window=('value','size'),
                  anomaly_count=('is_anomaly','sum'),
                  max_value_in_window=('value','max'),
                  global_P95=('global_P95','first')))

# Earliest anomaly timestamp per pair
earliest = (metric_window_candidates[metric_window_candidates['is_anomaly']==1]
            .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
            .min()
            .rename(columns={'timestamp':'earliest_anom_ts'}))

agg_pairs = agg_pairs.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# Format earliest timestamp to UTC+8 readable string
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

agg_pairs['earliest_anomaly_timestamp'] = agg_pairs['earliest_anom_ts'].apply(fmt_ts)

# 5) Prepare compact table sorted by earliest_anomaly_timestamp asc, then anomaly_count desc; limit to top 50
agg_pairs_sorted = agg_pairs.sort_values(by=['earliest_anom_ts','anomaly_count'], ascending=[True, False], na_position='last').reset_index(drop=True)

display_cols = ['cmdb_id','kpi_name','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_P95']
result_compact = agg_pairs_sorted[display_cols].copy()
result_compact['max_value_in_window'] = result_compact['max_value_in_window'].round(6)
result_compact['global_P95'] = result_compact['global_P95'].round(6)

# Cache full intermediates
top10_cmdbs_full = top10_cmdbs
top_kpis_per_cmdb_full = top_kpis_per_cmdb
metric_window_candidates_full = metric_window_candidates
agg_pairs_full = agg_pairs

# Return compact top 50
result_compact.head(50)

```
Out[7]:
```
```python
# Compose a compact plain-English summary based on cached results (top10_cmdbs_full, result_compact)
try:
    top10_cmdbs_full
    result_compact
except NameError:
    raise RuntimeError("Expected cached variables 'top10_cmdbs_full' and 'result_compact' not found.")

# Build short lists for the summary
top10_list = top10_cmdbs_full.values.tolist()
top10_txt = ", ".join([f"{row[0]}({int(row[1])})" for row in top10_list])

# Example metric anomaly rows (up to 5) for illustration
examples = result_compact.head(5)[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp']].values.tolist()
examples_txt = "; ".join([f"{r[0]}||{r[1]} (anomalies={int(r[2])}, earliest={r[3]})" for r in examples])

summary = (
    "Mapping trace-impacted components to metric anomalies (plain English):\n\n"
    f"- Top 10 components from trace anomalies (frequency): {top10_txt}.\n\n"
    "- For these components, I inspected metric KPIs related to CPU, memory, network, I/O, disk, load, and app latency.\n"
    "- Many metric pairs flagged as 'anomalies' occur at 2022-03-20 15:30:00 (UTC+8) with anomaly_count = 30; these often arise because global P95 == value (or P95 == 0),\n"
    "  which makes counters or static metrics appear as full-window anomalies (likely non-actionable).\n"
    "- However, there are non-trivial signals too (e.g., memory_cache, memory_failcnt, cpu_load_average, fs_io metrics) with non-zero P95 or differing max values.\n\n"
    "Examples (component||kpi_name with anomaly_count and earliest anomaly):\n"
    f"{examples_txt}\n\n"
    "Interpretation and next steps:\n"
    "1) The trace-driven top components (listed above) are good targets for focused metric correlation at ~15:30 UTC+8.\n"
    "2) Prioritize dynamic KPIs with non-zero/global_P95 < max_value (memory_cache, memory_failcnt, CPU load, network/io rates) rather than constant-zero or start-time metrics.\n"
    "3) Correlate these metric anomalies (earliest timestamps) with the trace earliest_anomaly_timestamp to identify likely root-cause nodes/pods and the precise failure windows.\n\n"
    "I can now (a) filter the metric anomalies to only non-constant dynamic KPIs for these top components, or (b) correlate the earliest timestamps with logs/error entries—tell me which."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_P95
0   checkoutservice-0    container.node-5.container_cpu_load_average_10s                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
1   checkoutservice-0  container.node-5.container_fs_io_current./dev/...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
2   checkoutservice-0  container.node-5.container_fs_io_time_seconds....                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
3   checkoutservice-0  container.node-5.container_fs_io_time_weighted...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
4   checkoutservice-0          container.node-5.container_memory_failcnt                      30             30   2022-03-20 15:30:00+0800            1536501.0   1536501.0
..                ...                                                ...                     ...            ...                        ...                  ...         ...
45         frontend-1  container.node-5.container_fs_io_time_weighted...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
46         frontend-1            container.node-5.container_memory_cache                      30             30   2022-03-20 15:30:00+0800              65536.0     65536.0
47         frontend-1          container.node-5.container_memory_failcnt                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
48         frontend-1  container.node-5.container_memory_failures.con...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0
49         frontend-1  container.node-5.container_memory_failures.hie...                      30             30   2022-03-20 15:30:00+0800                  0.0         0.0

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse cached variables from previous steps
try:
    top10_cmdbs_full
    result_compact
    agg_pairs_full
except NameError as e:
    raise RuntimeError("Required cached variables not found in kernel: top10_cmdbs_full, result_compact, agg_pairs_full") from e

tz = pytz.timezone('Asia/Shanghai')

# 1) Top 5 components by frequency from trace-anomaly frequency table
top5_cmdbs = top10_cmdbs_full.head(5)['cmdb_id'].tolist()

# 2) Start from the cached metric-anomaly mapping (agg_pairs_full has numeric earliest_anom_ts)
df = agg_pairs_full.copy()

# Ensure numeric columns
df['anomaly_count'] = pd.to_numeric(df['anomaly_count'], errors='coerce').fillna(0).astype(int)
df['max_value_in_window'] = pd.to_numeric(df['max_value_in_window'], errors='coerce')
df['global_P95'] = pd.to_numeric(df['global_P95'], errors='coerce')

# Filter: cmdb_id in top5, anomaly_count>0, and max_value_in_window > global_P95
mask_base = (
    df['cmdb_id'].isin(top5_cmdbs) &
    (df['anomaly_count'] > 0) &
    (df['max_value_in_window'] > df['global_P95'])
)
df = df[mask_base].copy()

# 3) Further restrict to kpi_name containing specified keywords (case-insensitive)
keywords = ['cpu','memory','memory_rss','mem','load','io','fs_io','io_time','network','net','disk','app.http','app.grpc','duration']
pattern = '(' + '|'.join([re.escape(k) for k in sorted(set(keywords), key=len, reverse=True)]) + ')'
df = df[df['kpi_name'].astype(str).str.contains(pattern, case=False, na=False, regex=True)].copy()

# 4) Compute breach_ratio:
# If global_P95 > 0: (max - P95) / P95 else use (max_value_in_window)
df['breach_ratio'] = np.where(
    df['global_P95'] > 0,
    (df['max_value_in_window'] - df['global_P95']) / df['global_P95'],
    df['max_value_in_window']
)

# 5) Prepare compact table sorted by earliest_anom_ts asc, then breach_ratio desc; limit to top 20
# Format earliest timestamp to UTC+8 string
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

df['earliest_anomaly_timestamp'] = df['earliest_anom_ts'].apply(fmt_ts)

# Select and round numeric columns for compactness
display_cols = ['cmdb_id','kpi_name','total_points_in_window','anomaly_count',
                'earliest_anomaly_timestamp','max_value_in_window','global_P95','breach_ratio']
df_display = df[display_cols].copy()
df_display['max_value_in_window'] = df_display['max_value_in_window'].round(6)
df_display['global_P95'] = df_display['global_P95'].round(6)
df_display['breach_ratio'] = df_display['breach_ratio'].round(6)

df_display_sorted = df_display.sort_values(by=['earliest_anomaly_timestamp','breach_ratio'],
                                           ascending=[True, False],
                                           na_position='last').reset_index(drop=True)

# Cache full filtered dataframe for next steps
focused_metric_anomalies_full = df.copy()

# Compact output: top 20 rows
focused_top20_display = df_display_sorted.head(20)

focused_top20_display

```
Out[8]:
```
```python
summary = (
    "Focused metric-anomaly summary (plain English):\n\n"
    "1) Top 5 trace-impacted components examined: checkoutservice-1, frontend-2, frontend-0, checkoutservice-0, frontend-1.\n\n"
    "2) Key metric anomalies (filtered to dynamic KPIs & non-constant breaches):\n"
    "- Memory failure indicators (container_memory_failures.*) on checkoutservice-1 and checkoutservice-0 (earliest ~2022-03-20 15:30–15:33+08). "
    "These show breach_ratio ≈ 0.11–0.82 (up to ~82%), indicating notable increases over the pair-specific P95.\n"
    "- container_memory_rss increases on frontend pods (frontend-2, frontend-0, frontend-1) with small but measurable breach_ratios (~0.001–0.006), earliest ~15:30–15:40+08.\n"
    "- CPU-related signals (cpu_cfs_throttled, cpu_cfs_periods, cpu_system_seconds, cpu_usage_seconds) appear on frontend and checkoutservice pods with breach_ratios from ~0.015 to ~0.82 and earliest timestamps ~15:33–15:40+08.\n\n"
    "3) Timing: many anomalies begin at or near 2022-03-20 15:30:00 (UTC+8) with further notable events through ~15:40–15:41. This aligns with the trace anomalies onset (~15:30) identified earlier.\n\n"
    "4) Interpretation: the strongest, actionable signals are memory-failure-related metrics on checkoutservice pods and CPU throttling/usage on frontend/checkoutservice pods. "
    "These suggest resource pressure (memory issues and CPU contention/throttling) around the incident start time rather than purely static counters.\n\n"
    "5) Recommended next steps:\n"
    "- Correlate these earliest metric breach timestamps with pod logs and Kubernetes events for the listed pods (checkoutservice-*, frontend-*) around 15:30–15:40+08.\n"
    "- Check node-level metrics for the nodes hosting these pods (CPU saturation, memory pressure, disk I/O or swap activity) and any pod restarts or OOM/Kill events.\n"
    "- Prioritize investigation of checkoutservice pods (memory_failures) and frontend pods (CPU throttling + memory_rss) as candidate root causes for the latency/trace anomalies.\n\n"
    "If you want, I can now (a) extract the raw metric time series for these top (cmdb_id,kpi) pairs across a wider time window for detailed inspection, or (b) correlate with error_logs.csv entries and Kubernetes events."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_P95  breach_ratio
0   checkoutservice-1  container.node-5.container_memory_failures.con...                      30              2   2022-03-20 15:30:00+0800         1.473500e+03  1.331125e+03      0.106958
1   checkoutservice-1  container.node-5.container_memory_failures.hie...                      30              2   2022-03-20 15:30:00+0800         1.473500e+03  1.331125e+03      0.106958
2          frontend-2              container.node-5.container_memory_rss                      30              2   2022-03-20 15:30:00+0800         2.006426e+07  2.003466e+07      0.001477
3          frontend-0              container.node-5.container_memory_rss                      30              2   2022-03-20 15:32:00+0800         1.950720e+07  1.937971e+07      0.006578
4   checkoutservice-0  container.node-5.container_memory_failures.con...                      30              2   2022-03-20 15:33:00+0800         5.000000e-01  2.750000e-01      0.818182
5   checkoutservice-0  container.node-5.container_memory_failures.hie...                      30              2   2022-03-20 15:33:00+0800         5.000000e-01  2.750000e-01      0.818182
6          frontend-0  container.node-5.container_cpu_cfs_throttled_s...                      30              2   2022-03-20 15:33:00+0800         5.050000e-04  2.780000e-04      0.818182
7   checkoutservice-0  container.node-5.container_cpu_cfs_throttled_p...                      30              2   2022-03-20 15:35:00+0800         4.500000e+00  3.550000e+00      0.267606
8   checkoutservice-0      container.node-5.container_cpu_system_seconds                      30              2   2022-03-20 15:35:00+0800         6.500000e-02  6.050000e-02      0.074380
9          frontend-1  container.node-5.container_memory_failures.con...                      30              2   2022-03-20 15:36:00+0800         4.106500e+03  3.568925e+03      0.150627
10         frontend-1  container.node-5.container_memory_failures.hie...                      30              2   2022-03-20 15:36:00+0800         4.106500e+03  3.568925e+03      0.150627
11  checkoutservice-1         container.node-5.container_cpu_cfs_periods                      30              2   2022-03-20 15:36:00+0800         3.250000e+01  3.005000e+01      0.081531
12         frontend-1         container.node-5.container_cpu_cfs_periods                      30              2   2022-03-20 15:36:00+0800         2.305000e+02  2.265500e+02      0.017435
13         frontend-1       container.node-5.container_cpu_usage_seconds                      30              2   2022-03-20 15:36:00+0800         7.175720e-01  7.063220e-01      0.015927
14         frontend-2         container.node-5.container_cpu_cfs_periods                      30              2   2022-03-20 15:38:00+0800         2.535000e+02  2.484250e+02      0.020429
15  checkoutservice-1      container.node-5.container_cpu_system_seconds                      30              5   2022-03-20 15:40:00+0800         6.500000e-02  5.500000e-02      0.181818
16  checkoutservice-1        container.node-5.container_cpu_user_seconds                      30              2   2022-03-20 15:40:00+0800         7.000000e-02  6.775000e-02      0.033210
17  checkoutservice-0              container.node-5.container_memory_rss                      30              2   2022-03-20 15:40:00+0800         1.755955e+07  1.745777e+07      0.005830
18         frontend-1              container.node-5.container_memory_rss                      30              2   2022-03-20 15:40:00+0800         2.094285e+07  2.088714e+07      0.002667
19         frontend-0         container.node-5.container_cpu_cfs_periods                      30              2   2022-03-20 15:41:00+0800         2.660000e+02  2.539500e+02      0.047450```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse cached dataframes if present; otherwise load (safe-guard)
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv('error_logs.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Top 5 candidate components (from prior cached table top10_cmdbs_full)
try:
    top10_cmdbs_full
    top5_components = top10_cmdbs_full.head(5)['cmdb_id'].tolist()
except Exception:
    # Fallback: if not present, use last focused components from earlier step
    top5_components = ['checkoutservice-1','frontend-2','frontend-0','checkoutservice-0','frontend-1']

# Time window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())

# Helper to format ts -> UTC+8 string
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# 1) Filter error_logs.csv to window and top5 components; cache full filtered table
error_filtered = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] < end_ts) &
    (error_logs_df['cmdb_id'].isin(top5_components))
].copy()

# For each component compute total_row_count, earliest_ts, latest_ts, up to 5 distinct message samples truncated to 200 chars
# Prepare messages: dropna, unique, truncated
def collect_messages(subdf, max_samples=5):
    msgs = subdf['message'].dropna().astype(str).unique()
    msgs_trunc = [m[:200] for m in msgs][:max_samples]
    return " || ".join(msgs_trunc) if len(msgs_trunc)>0 else None

err_grouped = (error_filtered
               .groupby('cmdb_id', dropna=False)
               .agg(total_row_count=('message','size'),
                    earliest_ts=('timestamp','min'),
                    latest_ts=('timestamp','max'))
               .reset_index())

# Attach message samples
err_grouped['message_samples'] = err_grouped['cmdb_id'].apply(
    lambda cid: collect_messages(error_filtered[error_filtered['cmdb_id']==cid])
)

# Format timestamps
err_grouped['earliest_timestamp'] = err_grouped['earliest_ts'].apply(fmt_ts)
err_grouped['latest_timestamp'] = err_grouped['latest_ts'].apply(fmt_ts)

# Keep only requested columns and order
error_summary_df = err_grouped[['cmdb_id','total_row_count','earliest_timestamp','latest_timestamp','message_samples']].copy()

# Ensure rows exist for components with zero rows (include them with zeros/nulls)
missing_err = [c for c in top5_components if c not in error_summary_df['cmdb_id'].values]
if missing_err:
    missing_df = pd.DataFrame({
        'cmdb_id': missing_err,
        'total_row_count': 0,
        'earliest_timestamp': [None]*len(missing_err),
        'latest_timestamp': [None]*len(missing_err),
        'message_samples':[None]*len(missing_err)
    })
    error_summary_df = pd.concat([error_summary_df, missing_df], ignore_index=True)

# Sort by cmdb_id for compactness
error_summary_df = error_summary_df.set_index('cmdb_id').loc[top5_components].reset_index()

# 2) Filter log.csv to window and top5 components; cache full filtered table
log_filtered = log_df[
    (log_df['timestamp'] >= start_ts) & (log_df['timestamp'] < end_ts) &
    (log_df['cmdb_id'].isin(top5_components))
].copy()

# For each component compute total_row_count (in window), sum_of_log_error_count (sum value where log_name=='log.error_count'),
# max_log_error_count, earliest_timestamp_where_log_error_count>0 (or null)
# Ensure numeric values
log_filtered['value'] = pd.to_numeric(log_filtered['value'], errors='coerce')

# Aggregate general counts
log_grouped = (log_filtered
               .groupby('cmdb_id', dropna=False)
               .agg(total_row_count=('value','size'))
               .reset_index())

# Compute sums and maxes for log.error_count specifically
error_count_df = log_filtered[log_filtered['log_name']=='log.error_count'].copy()
error_count_agg = (error_count_df
                   .groupby('cmdb_id', dropna=False)
                   .agg(sum_log_error_count=('value','sum'),
                        max_log_error_count=('value','max'))
                   .reset_index())

# Earliest timestamp where log.error_count > 0
ec_positive = error_count_df[error_count_df['value'] > 0].copy()
earliest_ec = (ec_positive.groupby('cmdb_id', dropna=False)['timestamp']
               .min()
               .reset_index()
               .rename(columns={'timestamp':'earliest_errorcount_ts'}))

# Merge together
log_summary = log_grouped.merge(error_count_agg, on='cmdb_id', how='left').merge(earliest_ec, on='cmdb_id', how='left')

# Fill NaNs for components with no error_count rows
log_summary['sum_log_error_count'] = log_summary['sum_log_error_count'].fillna(0)
log_summary['max_log_error_count'] = log_summary['max_log_error_count'].where(log_summary['max_log_error_count'].notna(), None)

log_summary['earliest_errorcount_timestamp'] = log_summary['earliest_errorcount_ts'].apply(lambda x: fmt_ts(x) if pd.notna(x) else None)

# Keep only requested columns
log_summary_df = log_summary[['cmdb_id','total_row_count','sum_log_error_count','max_log_error_count','earliest_errorcount_timestamp']].copy()

# Ensure all top5 components present
missing_log = [c for c in top5_components if c not in log_summary_df['cmdb_id'].values]
if missing_log:
    missing2 = pd.DataFrame({
        'cmdb_id': missing_log,
        'total_row_count': 0,
        'sum_log_error_count': 0,
        'max_log_error_count': [None]*len(missing_log),
        'earliest_errorcount_timestamp':[None]*len(missing_log)
    })
    log_summary_df = pd.concat([log_summary_df, missing2], ignore_index=True)

# Order rows by the top5 components
log_summary_df = log_summary_df.set_index('cmdb_id').loc[top5_components].reset_index()

# Cache full filtered tables for further steps
error_filtered_full = error_filtered
log_filtered_full = log_filtered
error_summary_full = err_grouped
log_summary_full = log_summary

# Return compact summaries (error_summary_df, log_summary_df)
error_summary_df, log_summary_df

```
Out[9]:
```
Summary (plain English):

- Error logs (error_logs.csv) within 2022-03-20 15:30–16:00 (UTC+8):
  - frontend-0: 706 error-log rows, earliest 2022-03-20 15:30:00, latest 15:59:59. Example messages begin with: "severity: warning, message: failed to retrieve..." (up to 5 distinct samples returned).
  - frontend-1: 580 error-log rows, earliest 15:30:02, latest 15:59:58. Similar sample messages.
  - frontend-2: 702 error-log rows, earliest 15:30:01, latest 15:59:59. Similar sample messages.
  - checkoutservice-0: no error_log rows in the window.
  - checkoutservice-1: no error_log rows in the window.

- Aggregated log metrics (log.csv) in the same window for the same components:
  - frontend-0: 60 log rows, sum(log.error_count)=706, max(log.error_count)=30, earliest timestamp with log.error_count>0 = 2022-03-20 15:30:00.
  - frontend-1: 60 log rows, sum(log.error_count)=580, max=23, earliest >0 = 15:30:00.
  - frontend-2: 60 log rows, sum(log.error_count)=702, max=30, earliest >0 = 15:30:00.
  - checkoutservice-0 and checkoutservice-1: 60 log rows each, but sum(log.error_count)=0 and no timestamps with log.error_count>0.

- Interpretation:
  - The three frontend pods (frontend-0/1/2) show heavy error activity that starts at the incident onset (~15:30). Error messages are primarily warnings like "failed to retrieve..." — these should be inspected in full to identify failing downstream calls or resource issues.
  - Checkoutservice pods show metric signs of problems in prior metric analysis (memory/failure and CPU signals) but did not emit error_log entries or log.error_count > 0 in this window. This means checkoutservice issues may be silent in logs (manifesting via metrics) or logs are captured elsewhere.

- Recommended next steps:
  1. Inspect full error log message texts for frontend-0/1/2 around 15:30 to find failed downstream calls, stack traces, or target services they cannot reach.
  2. Correlate frontend error timing (15:30) with the metric anomalies we identified (memory_failures, CPU throttling, memory_rss) on frontend and checkoutservice pods to determine direction (did resource pressure in checkoutservice cause frontend errors, or did frontend errors increase load and trigger resource pressure).
  3. Check pod events and Kubernetes logs for OOMKills, restarts, or network errors on nodes hosting those pods around 15:30.
  4. If desired, I can extract the full (untruncated) error messages, or pull the raw log rows and metric time series for specific pods for deeper correlation.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  total_row_count        earliest_timestamp          latest_timestamp                                    message_samples
0         frontend-0              706  2022-03-20 15:30:00+0800  2022-03-20 15:59:59+0800  severity: warning, message: failed to retrieve...
1         frontend-1              580  2022-03-20 15:30:02+0800  2022-03-20 15:59:58+0800  severity: warning, message: failed to retrieve...
2         frontend-2              702  2022-03-20 15:30:01+0800  2022-03-20 15:59:59+0800  severity: warning, message: failed to retrieve...
3  checkoutservice-0                0                      None                      None                                               None
4  checkoutservice-1                0                      None                      None                                               None,              cmdb_id  total_row_count  sum_log_error_count  max_log_error_count earliest_errorcount_timestamp
0         frontend-0               60                706.0                 30.0      2022-03-20 15:30:00+0800
1         frontend-1               60                580.0                 23.0      2022-03-20 15:30:00+0800
2         frontend-2               60                702.0                 30.0      2022-03-20 15:30:00+0800
3  checkoutservice-0               60                  0.0                  0.0                          None
4  checkoutservice-1               60                  0.0                  0.0                          None)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
import re

# Reuse existing error_logs_df if present, otherwise load
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv('error_logs.csv')

# Time window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())

# Target frontends
frontends = ['frontend-0','frontend-1','frontend-2']

# Helper to format timestamp to UTC+8 string
def fmt_ts_col(ts_series):
    return pd.to_datetime(ts_series.astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Filter rows to window and each frontend; cache full filtered tables
err_frontend0_df = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] < end_ts) &
    (error_logs_df['cmdb_id'] == 'frontend-0')
].copy()

err_frontend1_df = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] < end_ts) &
    (error_logs_df['cmdb_id'] == 'frontend-1')
].copy()

err_frontend2_df = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] < end_ts) &
    (error_logs_df['cmdb_id'] == 'frontend-2')
].copy()

# Prepare up-to-first-20 rows per frontend sorted by timestamp ascending and include readable timestamp + full message
frontend0_head = err_frontend0_df.sort_values('timestamp', ascending=True).head(20).copy()
if not frontend0_head.empty:
    frontend0_head['timestamp'] = fmt_ts_col(frontend0_head['timestamp'])
frontend0_head = frontend0_head[['timestamp','cmdb_id','message']]

frontend1_head = err_frontend1_df.sort_values('timestamp', ascending=True).head(20).copy()
if not frontend1_head.empty:
    frontend1_head['timestamp'] = fmt_ts_col(frontend1_head['timestamp'])
frontend1_head = frontend1_head[['timestamp','cmdb_id','message']]

frontend2_head = err_frontend2_df.sort_values('timestamp', ascending=True).head(20).copy()
if not frontend2_head.empty:
    frontend2_head['timestamp'] = fmt_ts_col(frontend2_head['timestamp'])
frontend2_head = frontend2_head[['timestamp','cmdb_id','message']]

# Keywords to count (case-insensitive)
keywords = ['oom','oomkill','out of memory','killed','failed to retrieve',
            'connection refused','timeout','refused','exception','error']

# Function to count matches per keyword for a dataframe
def keyword_counts_for_df(df, cmdb_id):
    total_rows = df.shape[0]
    rows = []
    messages = df['message'].astype(str).fillna('')
    for kw in keywords:
        # escape special regex characters except spaces so we treat phrases literally
        pat = re.escape(kw)
        cnt = messages.str.contains(pat, case=False, na=False, regex=True).sum()
        rows.append({'cmdb_id': cmdb_id, 'total_rows_in_window': total_rows, 'keyword': kw, 'match_count': int(cnt)})
    return pd.DataFrame(rows)

# Build keyword-count table for each frontend and combine
kc0 = keyword_counts_for_df(err_frontend0_df, 'frontend-0')
kc1 = keyword_counts_for_df(err_frontend1_df, 'frontend-1')
kc2 = keyword_counts_for_df(err_frontend2_df, 'frontend-2')

keyword_counts_df = pd.concat([kc0,kc1,kc2], ignore_index=True)

# Cache full filtered tables (already stored as err_frontend*_df and keyword_counts_df)
error_filtered_frontends_full = pd.concat([err_frontend0_df.assign(source='frontend-0'),
                                           err_frontend1_df.assign(source='frontend-1'),
                                           err_frontend2_df.assign(source='frontend-2')], ignore_index=True)

# Return compact outputs: up to 20 full-message rows per frontend, and the keyword-count table
frontend0_head, frontend1_head, frontend2_head, keyword_counts_df

```
Out[10]:
```
```python
summary = (
    "Compact findings from error_logs for frontend-0/1/2 (2022-03-20 15:30–16:00, UTC+8):\n\n"
    "1) High-level counts and time range:\n"
    "- frontend-0: 706 error-log rows (earliest 2022-03-20 15:30:00, latest 15:59:59).\n"
    "- frontend-1: 580 error-log rows (earliest 2022-03-20 15:30:02, latest 15:59:58).\n"
    "- frontend-2: 702 error-log rows (earliest 2022-03-20 15:30:01, latest 15:59:59).\n\n"
    "2) Message examples and pattern:\n"
    "- The sampled messages (first 20 per pod) are repeatedly: \"severity: warning, message: failed to retrieve...\" with timestamps every few seconds starting at ~15:30. Full messages are cached in memory for deeper inspection.\n\n"
    "3) Keyword scan (counts within the window):\n"
    "- The phrase \"failed to retrieve\" matches essentially all error rows for each frontend (frontend-0: 706 matches; frontend-1: 580; frontend-2: 702).\n"
    "- No matches were found for OOM/OOMKill/out of memory/killed/connection refused/timeout/refused/exception in these frontend logs (all those keywords show 0 matches).\n\n"
    "4) Interpretation:\n"
    "- Frontend pods show heavy, repeated warnings about \"failed to retrieve\" starting at the incident onset (~15:30). This indicates frontends are repeatedly failing to fetch some downstream resource or API (not necessarily OOM or OS-level kills).\n"
    "- The absence of 'oom'/'oomkill' keywords suggests these frontend errors are not due to container OOM kills. Combined with earlier metric signals (memory_failures and CPU throttling seen on checkoutservice and frontend pods), this points toward request/response failures likely caused by degraded downstream service(s) or network/connectivity/resource pressure.\n\n"
    "5) Immediate recommended actions:\n"
    "- Inspect the full (uncut) \"failed to retrieve\" messages to identify the target service/endpoint or stack trace (cached rows are available).\n"
    "- Correlate the exact failed-call timestamps (~15:30 onward) with metric anomalies we found (memory_failures on checkoutservice*, CPU throttling on frontend/checkoutservice) and with Kubernetes events for pod restarts or node issues.\n"
    "- Check network connectivity and service availability for the backend services that frontends call (timeouts, connection errors at the network layer, or overloaded backend causing failed retrievals).\n\n"
    "If you want, I can now (A) show the full, untruncated error messages for these frontends, (B) extract the raw metric time series for the implicated pods around the exact failure seconds, or (C) search logs for the backend service names mentioned in the \"failed to retrieve\" messages."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp     cmdb_id                                            message
0   2022-03-20 15:30:00+0800  frontend-0  severity: warning, message: failed to retrieve...
4   2022-03-20 15:30:02+0800  frontend-0  severity: warning, message: failed to retrieve...
7   2022-03-20 15:30:06+0800  frontend-0  severity: warning, message: failed to retrieve...
8   2022-03-20 15:30:09+0800  frontend-0  severity: warning, message: failed to retrieve...
11  2022-03-20 15:30:13+0800  frontend-0  severity: warning, message: failed to retrieve...
15  2022-03-20 15:30:16+0800  frontend-0  severity: warning, message: failed to retrieve...
18  2022-03-20 15:30:18+0800  frontend-0  severity: warning, message: failed to retrieve...
23  2022-03-20 15:30:23+0800  frontend-0  severity: warning, message: failed to retrieve...
27  2022-03-20 15:30:27+0800  frontend-0  severity: warning, message: failed to retrieve...
30  2022-03-20 15:30:29+0800  frontend-0  severity: warning, message: failed to retrieve...
34  2022-03-20 15:30:31+0800  frontend-0  severity: warning, message: failed to retrieve...
38  2022-03-20 15:30:35+0800  frontend-0  severity: warning, message: failed to retrieve...
40  2022-03-20 15:30:38+0800  frontend-0  severity: warning, message: failed to retrieve...
43  2022-03-20 15:30:41+0800  frontend-0  severity: warning, message: failed to retrieve...
44  2022-03-20 15:30:41+0800  frontend-0  severity: warning, message: failed to retrieve...
47  2022-03-20 15:30:43+0800  frontend-0  severity: warning, message: failed to retrieve...
49  2022-03-20 15:30:45+0800  frontend-0  severity: warning, message: failed to retrieve...
52  2022-03-20 15:30:48+0800  frontend-0  severity: warning, message: failed to retrieve...
54  2022-03-20 15:30:49+0800  frontend-0  severity: warning, message: failed to retrieve...
57  2022-03-20 15:30:51+0800  frontend-0  severity: warning, message: failed to retrieve...,                    timestamp     cmdb_id                                            message
2   2022-03-20 15:30:02+0800  frontend-1  severity: warning, message: failed to retrieve...
9   2022-03-20 15:30:10+0800  frontend-1  severity: warning, message: failed to retrieve...
13  2022-03-20 15:30:14+0800  frontend-1  severity: warning, message: failed to retrieve...
17  2022-03-20 15:30:17+0800  frontend-1  severity: warning, message: failed to retrieve...
19  2022-03-20 15:30:18+0800  frontend-1  severity: warning, message: failed to retrieve...
24  2022-03-20 15:30:24+0800  frontend-1  severity: warning, message: failed to retrieve...
26  2022-03-20 15:30:26+0800  frontend-1  severity: warning, message: failed to retrieve...
29  2022-03-20 15:30:28+0800  frontend-1  severity: warning, message: failed to retrieve...
32  2022-03-20 15:30:30+0800  frontend-1  severity: warning, message: failed to retrieve...
35  2022-03-20 15:30:32+0800  frontend-1  severity: warning, message: failed to retrieve...
36  2022-03-20 15:30:34+0800  frontend-1  severity: warning, message: failed to retrieve...
39  2022-03-20 15:30:36+0800  frontend-1  severity: warning, message: failed to retrieve...
41  2022-03-20 15:30:40+0800  frontend-1  severity: warning, message: failed to retrieve...
46  2022-03-20 15:30:41+0800  frontend-1  severity: warning, message: failed to retrieve...
50  2022-03-20 15:30:47+0800  frontend-1  severity: warning, message: failed to retrieve...
56  2022-03-20 15:30:51+0800  frontend-1  severity: warning, message: failed to retrieve...
59  2022-03-20 15:30:53+0800  frontend-1  severity: warning, message: failed to retrieve...
62  2022-03-20 15:30:56+0800  frontend-1  severity: warning, message: failed to retrieve...
65  2022-03-20 15:30:59+0800  frontend-1  severity: warning, message: failed to retrieve...
67  2022-03-20 15:31:00+0800  frontend-1  severity: warning, message: failed to retrieve...,                    timestamp     cmdb_id                                            message
1   2022-03-20 15:30:01+0800  frontend-2  severity: warning, message: failed to retrieve...
3   2022-03-20 15:30:02+0800  frontend-2  severity: warning, message: failed to retrieve...
5   2022-03-20 15:30:05+0800  frontend-2  severity: warning, message: failed to retrieve...
6   2022-03-20 15:30:06+0800  frontend-2  severity: warning, message: failed to retrieve...
10  2022-03-20 15:30:11+0800  frontend-2  severity: warning, message: failed to retrieve...
12  2022-03-20 15:30:14+0800  frontend-2  severity: warning, message: failed to retrieve...
14  2022-03-20 15:30:15+0800  frontend-2  severity: warning, message: failed to retrieve...
16  2022-03-20 15:30:16+0800  frontend-2  severity: warning, message: failed to retrieve...
20  2022-03-20 15:30:19+0800  frontend-2  severity: warning, message: failed to retrieve...
21  2022-03-20 15:30:20+0800  frontend-2  severity: warning, message: failed to retrieve...
22  2022-03-20 15:30:22+0800  frontend-2  severity: warning, message: failed to retrieve...
25  2022-03-20 15:30:25+0800  frontend-2  severity: warning, message: failed to retrieve...
28  2022-03-20 15:30:27+0800  frontend-2  severity: warning, message: failed to retrieve...
31  2022-03-20 15:30:29+0800  frontend-2  severity: warning, message: failed to retrieve...
33  2022-03-20 15:30:31+0800  frontend-2  severity: warning, message: failed to retrieve...
37  2022-03-20 15:30:34+0800  frontend-2  severity: warning, message: failed to retrieve...
42  2022-03-20 15:30:40+0800  frontend-2  severity: warning, message: failed to retrieve...
45  2022-03-20 15:30:41+0800  frontend-2  severity: warning, message: failed to retrieve...
48  2022-03-20 15:30:44+0800  frontend-2  severity: warning, message: failed to retrieve...
51  2022-03-20 15:30:47+0800  frontend-2  severity: warning, message: failed to retrieve...,        cmdb_id  total_rows_in_window             keyword  match_count
0   frontend-0                   706                 oom            0
1   frontend-0                   706             oomkill            0
2   frontend-0                   706       out of memory            0
3   frontend-0                   706              killed            0
4   frontend-0                   706  failed to retrieve          706
..         ...                   ...                 ...          ...
25  frontend-2                   702  connection refused            0
26  frontend-2                   702             timeout            0
27  frontend-2                   702             refused            0
28  frontend-2                   702           exception            0
29  frontend-2                   702               error            0

[30 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse cached variables where available; else reconstruct minimal required ones
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_window (filtered-window metrics) exists
if 'metric_window' not in globals():
    # compute windowed metric_df
    try:
        metric_df
    except NameError:
        metric_df = pd.read_csv('metric.csv')
    start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
    end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())
    metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# Ensure global_p95_df exists (global thresholds for metrics)
if 'global_p95_df' not in globals():
    global_p95_df = (metric_df
                     .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
                     .quantile(0.95)
                     .rename(columns={'value':'global_P95'}))

# Ensure agg_full (per-pair window aggregates with max_value_in_window) exists; otherwise compute from metric_window
if 'agg_full' not in globals():
    agg_full = (metric_window
                .groupby(['cmdb_id','kpi_name'], as_index=False)
                .agg(max_value_in_window=('value','max'),
                     total_points_in_window=('value','size')))

# Dynamic KPI name pattern (case-insensitive)
dyn_keywords = ['cpu','memory','mem','load','io','fs_io','io_time','network','net','app.http','app.grpc','duration']
dyn_pat = '(' + '|'.join([re.escape(k) for k in sorted(set(dyn_keywords), key=len, reverse=True)]) + ')'

# 1) Metric anomaly events: rows in metric_window where value >= global_P95 AND pair's max_value_in_window > global_P95 AND kpi_name matches dynamic keywords
metric_win_merged = metric_window.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')
# attach pair max from agg_full if present
if 'max_value_in_window' in agg_full.columns:
    pair_max = agg_full[['cmdb_id','kpi_name','max_value_in_window']].drop_duplicates()
else:
    pair_max = (metric_window.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].max().rename(columns={'value':'max_value_in_window'}))
metric_win_merged = metric_win_merged.merge(pair_max, on=['cmdb_id','kpi_name'], how='left')

mask_metric_dynamic = metric_win_merged['kpi_name'].astype(str).str.contains(dyn_pat, case=False, na=False, regex=True)
mask_metric_anom = (metric_win_merged['value'] >= metric_win_merged['global_P95']) & (metric_win_merged['max_value_in_window'] > metric_win_merged['global_P95']) & mask_metric_dynamic
metric_events = metric_win_merged[mask_metric_anom].copy()

# Compute breach_ratio for metric events (if global_P95>0)
metric_events['breach_ratio'] = np.where(metric_events['global_P95'] > 0,
                                         (metric_events['value'] - metric_events['global_P95']) / metric_events['global_P95'],
                                         metric_events['value'])
metric_events['event_type'] = 'metric'
metric_events = metric_events[['timestamp','cmdb_id','kpi_name','value','global_P95','breach_ratio','event_type']].rename(columns={'kpi_name':'name'})

# 2) Trace anomaly events: use cached trace_window_top (merged with global_trace_p95_df and flagged earlier)
if 'trace_window_top' not in globals():
    # reconstruct trace window and merge p95
    try:
        trace_df
    except NameError:
        trace_df = pd.read_csv('trace.csv')
    global_trace_p95_df = (trace_df
                           .groupby(['cmdb_id','trace_name'], as_index=False)['value']
                           .quantile(0.95)
                           .rename(columns={'value':'global_P95'})
                           )
    start_ts = int(pd.Timestamp('2022-03-20 15:30:00', tz=tz).timestamp())
    end_ts = int(pd.Timestamp('2022-03-20 16:00:00', tz=tz).timestamp())
    trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)].copy()
    trace_window_top = trace_window.merge(global_trace_p95_df, on=['cmdb_id','trace_name'], how='left')
else:
    # ensure global_trace_p95_df exists
    if 'global_trace_p95_df' not in globals():
        global_trace_p95_df = trace_window_top[['cmdb_id','trace_name','global_P95']].drop_duplicates()

# trace name pattern and exclude trivial error_rate where global_P95==0
trace_keywords = ['duration','duration_p95','duration_mean','row_count','error_rate']
trace_pat = '(' + '|'.join([re.escape(k) for k in trace_keywords]) + ')'

mask_trace_name = trace_window_top['trace_name'].astype(str).str.contains(trace_pat, case=False, na=False, regex=True)
# select rows where value > global_P95 (strict) and match name pattern
mask_trace_anom = (trace_window_top['value'] > trace_window_top['global_P95']) & mask_trace_name
# exclude rows where trace_name contains 'error_rate' and global_P95 == 0 (trivial)
mask_error_rate = trace_window_top['trace_name'].str.contains('error_rate', case=False, na=False)
mask_trivial_error_rate = mask_error_rate & (trace_window_top['global_P95'] == 0)
trace_events = trace_window_top[mask_trace_anom & (~mask_trivial_error_rate)].copy()

# compute trace severity as delta = value - global_P95
trace_events['trace_delta'] = trace_events['value'] - trace_events['global_P95']
trace_events['event_type'] = 'trace'
trace_events = trace_events[['timestamp','cmdb_id','trace_name','value','global_P95','trace_delta','event_type']].rename(columns={'trace_name':'name'})

# 3) Combine metric and trace events into unified events dataframe
# Standardize columns: timestamp, cmdb_id, event_type, name, value, global_P95, severity
metric_events_std = metric_events.rename(columns={'breach_ratio':'severity'})
trace_events_std = trace_events.rename(columns={'trace_delta':'severity'})
events = pd.concat([metric_events_std, trace_events_std], ignore_index=True, sort=False)

# Group by timestamp (minute aligned already) and count events
events_grouped = (events
                  .groupby('timestamp', as_index=False)
                  .agg(total_event_count=('name','size'),
                       metric_event_count=('event_type', lambda s: (s=='metric').sum()),
                       trace_event_count=('event_type', lambda s: (s=='trace').sum()))
                  .sort_values('total_event_count', ascending=False)
                 )

# Format timestamp to UTC+8 string for display
events_grouped['timestamp_readable'] = pd.to_datetime(events_grouped['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')

top5_minutes = events_grouped.head(5)[['timestamp_readable','total_event_count','metric_event_count','trace_event_count']].rename(columns={'timestamp_readable':'timestamp'})

# 4) For the top 2 minutes, produce per-minute top-5 components by anomaly count and most severe anomaly for that component
top_minutes_list = events_grouped.head(2)['timestamp'].tolist()

per_minute_summaries = {}
for ts in top_minutes_list:
    ev_min = events[events['timestamp'] == ts].copy()
    # count per component
    comp_counts = ev_min.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'event_count'})
    comp_counts = comp_counts.sort_values('event_count', ascending=False).reset_index(drop=True)
    top5_comps = comp_counts.head(5)['cmdb_id'].tolist()
    # For each top component, find its most severe event in this minute
    rows = []
    for comp in top5_comps:
        comp_events = ev_min[ev_min['cmdb_id']==comp].copy()
        # choose most severe by severity numeric (NaN -> -inf)
        comp_events['severity_numeric'] = pd.to_numeric(comp_events['severity'], errors='coerce').fillna(-np.inf)
        idx = comp_events['severity_numeric'].idxmax()
        if pd.isna(idx):
            continue
        r = comp_events.loc[idx]
        if r['event_type']=='metric':
            # metric: report kpi_name (r['name']) and breach_ratio = severity
            rows.append({
                'cmdb_id': comp,
                'event_count': int(comp_events.shape[0]),
                'most_severe_type': 'metric',
                'most_severe_name': r['name'],
                'severity_metric': float(r['severity']),
                'value': float(r['value']),
                'global_P95': float(r['global_P95'])
            })
        else:
            # trace: report trace_name and delta = value - global_P95
            rows.append({
                'cmdb_id': comp,
                'event_count': int(comp_events.shape[0]),
                'most_severe_type': 'trace',
                'most_severe_name': r['name'],
                'severity_metric': float(r['severity']),
                'value': float(r['value']),
                'global_P95': float(r['global_P95'])
            })
    per_minute_summaries[ts] = pd.DataFrame(rows)

# Format per-minute timestamps for display keys
top_minutes_readable = [pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') for t in top_minutes_list]

# Cache full intermediate tables
metric_events_full = metric_events
trace_events_full = trace_events
events_full = events
events_grouped_full = events_grouped

# Compact outputs: top5_minutes and two per-minute top-5 component summaries (with timestamp context)
# Prepare display: pair each per-minute summary with its readable timestamp
minute1_summary = per_minute_summaries[top_minutes_list[0]] if len(top_minutes_list) > 0 else pd.DataFrame()
minute2_summary = per_minute_summaries[top_minutes_list[1]] if len(top_minutes_list) > 1 else pd.DataFrame()

# Add timestamp column to minute summaries for clarity
if not minute1_summary.empty:
    minute1_summary.insert(0, 'minute_timestamp', top_minutes_readable[0])
if not minute2_summary.empty:
    minute2_summary.insert(0, 'minute_timestamp', top_minutes_readable[1])

# Return compact results
top5_minutes, minute1_summary.head(5), minute2_summary.head(5)

```
Out[11]:
```
Summary of aggregated anomaly clusters (plain English):

Top 5 anomaly minutes (incident window 2022-03-20 15:30–16:00, UTC+8)
- 2022-03-20 15:49 — total events: 415 (metric: 325, trace: 90)
- 2022-03-20 15:58 — total events: 324 (metric: 294, trace: 30)
- 2022-03-20 15:51 — total events: 294 (metric: 243, trace: 51)
- 2022-03-20 15:41 — total events: 277 (metric: 221, trace: 56)
- 2022-03-20 15:53 — total events: 259 (metric: 189, trace: 70)

Top-2 minute clusters (most significant) — component-level summary (top 5 components each)

1) Minute = 2022-03-20 15:49 (415 events)
- frontend-1 (49 events)
  - Most severe: trace — trace.to_currencyservice-0.row_count
  - Severity: value - P95 = 125.00 - 118.55 = 6.45 (high row_count spike)
- frontend-0 (26 events)
  - Most severe: metric — container.node-5.container_cpu_system_seconds
  - Breach ratio ~ 0.0267 (small relative CPU system increase)
- frontend2-0 (26 events)
  - Most severe: trace — trace.self.row_count
  - Severity: value - P95 = 797.00 - 733.05 = 63.95 (large row_count spike)
- checkoutservice2-0 (24 events)
  - Most severe: metric — service-mesh destination metric (mesh.destination.frontend2.checkoutservice2...). Severity ~0.0596
- productcatalogservice-0 (23 events)
  - Most severe: trace — trace.from_frontend-2.row_count
  - Severity: value - P95 = 203 - 196 = 7 (row_count spike)

Interpretation for 15:49:
- This minute shows strong traffic/row_count spikes on frontends and some backend services (productcatalogservice), plus many metric anomalies (CPU/mesh). The largest severities are row_count spikes (frontends), indicating load surges or routing shifts at 15:49 with correlated metric degradations.

2) Minute = 2022-03-20 15:58 (324 events)
- checkoutservice-2 (43 events)
  - Most severe: metric — mesh.source.checkoutservice → currencyservice related metric
  - Severity: small positive delta (~0.0128)
- checkoutservice-0 (29 events)
  - Most severe: metric — container.node-5.container_cpu_cfs_throttled_p... (CPU throttling)
  - Breach ratio ~0.1268 (notable throttling)
- checkoutservice-1 (28 events)
  - Most severe: metric — mesh.destination.frontend.checkoutservice... (mesh metric)
  - Severity delta ~0.039
- frontend-2 (14 events)
  - Most severe: trace — trace.from_root.row_count
  - Severity: value - P95 = 128 - 127.55 = 0.45 (small row_count increase)
- currencyservice-2 (14 events)
  - Most severe: metric — container_memory_failures.con... 
  - Breach ratio ~0.0626 (memory-failure related increase)

Interpretation for 15:58:
- This cluster is dominated by checkoutservice pods and shows CPU throttling and mesh-related metric anomalies, plus memory-failure signals on currencyservice. It suggests resource contention or service-mesh issues affecting checkoutservice and downstream currencyservice at ~15:58.

Overall conclusions and recommended next steps:
- Two prominent anomaly clusters to investigate first: ~15:49 and ~15:58 (UTC+8). These minutes show concentrated metric + trace anomalies across frontends, checkoutservice pods, and some downstream services (productcatalogservice, currencyservice).
- Likely actionable root-cause candidates:
  - Frontend pods (large row_count spikes and repeated trace anomalies around 15:49).
  - Checkoutservice pods (CPU throttling and mesh-related anomalies around 15:58).
  - Currencyservice / productcatalogservice show correlated memory/row_count signals.
- Action items:
  1. Pull full metric time series (CPU, throttling, memory_failures, memory_rss, network I/O) for implicated pods and hosting nodes around 15:40–16:00 to confirm resource pressure.
  2. Inspect frontend full error messages and backend service logs precisely at 15:49 and 15:58 to identify failing endpoints or mesh errors.
  3. Check Kubernetes events (OOMKills, restarts), node-level metrics, and Istio/mesh logs near those two minutes.
  4. Prioritize remediation and deeper correlation starting with frontend and checkoutservice pods at the two minutes above.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  total_event_count  metric_event_count  trace_event_count
19  2022-03-20 15:49:00+0800                415                 325                 90
28  2022-03-20 15:58:00+0800                324                 294                 30
21  2022-03-20 15:51:00+0800                294                 243                 51
11  2022-03-20 15:41:00+0800                277                 221                 56
23  2022-03-20 15:53:00+0800                259                 189                 70,            minute_timestamp                  cmdb_id  event_count most_severe_type                                   most_severe_name  severity_metric   value  global_P95
0  2022-03-20 15:49:00+0800               frontend-1           49            trace               trace.to_currencyservice-0.row_count         6.450000  125.00   118.55000
1  2022-03-20 15:49:00+0800               frontend-0           26           metric      container.node-5.container_cpu_system_seconds         0.026654    0.26     0.25325
2  2022-03-20 15:49:00+0800              frontend2-0           26            trace                               trace.self.row_count        63.950000  797.00   733.05000
3  2022-03-20 15:49:00+0800       checkoutservice2-0           24           metric  mesh.destination.frontend2.checkoutservice2.is...         0.059603    4.00     3.77500
4  2022-03-20 15:49:00+0800  productcatalogservice-0           23            trace                    trace.from_frontend-2.row_count         7.000000  203.00   196.00000,            minute_timestamp            cmdb_id  event_count most_severe_type                                   most_severe_name  severity_metric        value  global_P95
0  2022-03-20 15:58:00+0800  checkoutservice-2           43           metric  mesh.source.checkoutservice.currencyservice.is...         0.012784     0.017825      0.0176
1  2022-03-20 15:58:00+0800  checkoutservice-0           29           metric  container.node-5.container_cpu_cfs_throttled_p...         0.126761     4.000000      3.5500
2  2022-03-20 15:58:00+0800  checkoutservice-1           28           metric  mesh.destination.frontend.checkoutservice.isti...         0.038961     6.000000      5.7750
3  2022-03-20 15:58:00+0800         frontend-2           14            trace                          trace.from_root.row_count         0.450000   128.000000    127.5500
4  2022-03-20 15:58:00+0800  currencyservice-2           14           metric  container.node-5.container_memory_failures.con...         0.062623  4075.000000   3834.8500)```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse cached variables (from previous steps); raise clear error if missing critical ones
required = ['events_full', 'metric_events_full', 'trace_events_full', 'error_filtered_full']
for v in required:
    if v not in globals():
        raise RuntimeError(f"Required cached variable '{v}' not found in kernel. Please run prior steps to produce it.")

tz = pytz.timezone('Asia/Shanghai')

# Target minutes (readable) and epoch timestamps (aligned to minute)
minutes_readable = ['2022-03-20 15:49:00+0800', '2022-03-20 15:58:00+0800']
minutes_ts = [int(pd.to_datetime(m).tz_convert(tz).timestamp()) for m in pd.to_datetime(minutes_readable)]

# Dynamic KPI keywords for metric filtering
dyn_keywords = ['cpu','memory','mem','load','io','fs_io','io_time','network','net','app.http','app.grpc','duration']
dyn_pat = re.compile('(' + '|'.join([re.escape(k) for k in sorted(set(dyn_keywords), key=len, reverse=True)]) + ')', flags=re.IGNORECASE)

# Trace name pattern and exclusion for trivial error_rate (global_P95==0)
trace_keywords = ['duration','duration_p95','duration_mean','row_count','error_rate']
trace_pat = re.compile('(' + '|'.join([re.escape(k) for k in trace_keywords]) + ')', flags=re.IGNORECASE)

# Short helper to format minute ts to readable string
def fmt_min(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# Use cached event tables
events = events_full.copy()
metric_events = metric_events_full.copy()
trace_events = trace_events_full.copy()
error_logs_window = error_filtered_full.copy()  # filtered to incident window earlier

# Ensure metric_events has breach_ratio column
if 'breach_ratio' not in metric_events.columns:
    metric_events['breach_ratio'] = np.where(metric_events['global_P95'] > 0,
                                             (metric_events['value'] - metric_events['global_P95']) / metric_events['global_P95'],
                                             metric_events['value'])

# Ensure trace_events has severity (value - global_P95) column
if 'severity' not in trace_events.columns:
    trace_events['severity'] = trace_events['value'] - trace_events['global_P95']

# Prepare outputs containers
top8_minutes = {}
details_minutes = {}

for ts in minutes_ts:
    # 1) Top 8 components by anomaly event count within this exact minute
    ev_at_min = events[events['timestamp'] == ts].copy()
    comp_counts = ev_at_min.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'event_count'})
    comp_counts = comp_counts.sort_values('event_count', ascending=False).reset_index(drop=True)
    top8 = comp_counts.head(8).copy()
    top8_minutes[ts] = top8[['cmdb_id','event_count']]

    # 2-4) For each top component collect metric, trace, and error log summaries at that minute
    rows = []
    for _, crow in top8.iterrows():
        comp = crow['cmdb_id']
        # Metric anomalies at minute for this component (dynamic KPIs only)
        m_ev = metric_events[(metric_events['timestamp']==ts) & (metric_events['cmdb_id']==comp)].copy()
        if not m_ev.empty:
            m_ev = m_ev[m_ev['name'].astype(str).apply(lambda x: bool(dyn_pat.search(x)))].copy()
        # Ensure breach_ratio exists
        if not m_ev.empty and 'breach_ratio' not in m_ev.columns:
            m_ev['breach_ratio'] = np.where(m_ev['global_P95'] > 0,
                                            (m_ev['value'] - m_ev['global_P95']) / m_ev['global_P95'],
                                            m_ev['value'])
        # Top 5 metric anomalies by breach_ratio desc
        if not m_ev.empty:
            m_top5 = m_ev.sort_values('breach_ratio', ascending=False).head(5)
            metric_summaries = [
                {
                    'kpi_name': r['name'],
                    'value': float(r['value']),
                    'global_P95': float(r['global_P95']),
                    'breach_ratio': float(r['breach_ratio'])
                } for _, r in m_top5.iterrows()
            ]
        else:
            metric_summaries = []

        # Trace anomalies at minute for this component (matching trace keywords), exclude trivial error_rate where global_P95==0
        t_ev = trace_events[(trace_events['timestamp']==ts) & (trace_events['cmdb_id']==comp)].copy()
        if not t_ev.empty:
            t_ev = t_ev[t_ev['name'].astype(str).apply(lambda x: bool(trace_pat.search(x)))].copy()
            # exclude trivial error_rate where global_P95 == 0
            t_ev = t_ev[~((t_ev['name'].str.contains('error_rate', case=False, na=False)) & (t_ev['global_P95']==0))].copy()
        # Ensure severity column exists
        if not t_ev.empty and 'severity' not in t_ev.columns:
            t_ev['severity'] = t_ev['value'] - t_ev['global_P95']
        # Top 5 trace anomalies by severity desc
        if not t_ev.empty:
            t_top5 = t_ev.sort_values('severity', ascending=False).head(5)
            trace_summaries = [
                {
                    'trace_name': r['name'],
                    'value': float(r['value']),
                    'global_P95': float(r['global_P95']),
                    'delta': float(r['severity'])
                } for _, r in t_top5.iterrows()
            ]
        else:
            trace_summaries = []

        # Error logs exactly at that minute for this component
        err_rows = error_logs_window[(error_logs_window['timestamp']==ts) & (error_logs_window['cmdb_id']==comp)].copy()
        err_count = int(err_rows.shape[0])
        msg_samples = err_rows['message'].astype(str).dropna().unique()[:3].tolist() if err_count > 0 else []

        rows.append({
            'minute_timestamp': fmt_min(ts),
            'cmdb_id': comp,
            'event_count': int(crow['event_count']),
            'metric_anomalies': metric_summaries,
            'trace_anomalies': trace_summaries,
            'error_log_count_in_minute': err_count,
            'error_message_samples': msg_samples
        })

    details_minutes[ts] = pd.DataFrame(rows)

# Prepare compact return tables for two minutes
m1_ts = minutes_ts[0]
m2_ts = minutes_ts[1]

top8_minute1 = top8_minutes[m1_ts].reset_index(drop=True)
details_minute1 = details_minutes[m1_ts].reset_index(drop=True)

top8_minute2 = top8_minutes[m2_ts].reset_index(drop=True)
details_minute2 = details_minutes[m2_ts].reset_index(drop=True)

# Cache full intermediates for further analysis
top8_minutes_full = top8_minutes
details_minutes_full = details_minutes

# Return compact summaries: top-8 lists and per-component detailed mini-summaries for both minutes
top8_minute1, details_minute1, top8_minute2, details_minute2

```
Out[12]:
```
Summary corroboration for the two target minutes (UTC+8):

Minute 2022-03-20 15:49
- Top-8 components by anomaly events (count):
  1. frontend-1 (49)
  2. frontend-0 (26)
  3. frontend2-0 (26)
  4. checkoutservice2-0 (24)
  5. productcatalogservice-0 (23)
  6. productcatalogservice-2 (23)
  7. checkoutservice-0 (19)
  8. productcatalogservice-1 (15)

- Metric evidence (most severe per component, up to 5):
  - Frontends: CPU/system metrics (e.g., container.node-5.container_cpu_system_seconds) and some mesh-related metrics flagged — small-to-moderate breach ratios but many events.
  - checkoutservice2-0 & productcatalogservice-* : mesh-related metrics (service-mesh destination/source metrics) and other container KPIs showing breaches.
  - Overall metric pattern: many mesh and CPU-related anomalies at this minute.

- Trace evidence (most severe per component):
  - Frontends and productcatalogservice show large row_count spikes (trace.row_count) and service-to-service trace anomalies (e.g., trace.to_currencyservice-0.row_count, trace.self.row_count, trace.from_frontend-2.row_count).
  - These trace spikes indicate sudden traffic/throughput differences or request bursts at 15:49.

- Error logs at exact minute:
  - No error_logs entries exactly at 15:49 for these top components (error_log_count_in_minute = 0 for each).

Interpretation for 15:49:
- Strong, concentrated traffic/trace row_count spikes on frontends and productcatalogservice plus mesh/CPU metric anomalies suggest a load surge or routing/mesh issue at 15:49. Frontends show the largest event counts; traces indicate increased request rates to backend services (productcatalogservice, currencyservice, etc.). No direct error-log entries at the exact minute, but earlier/nearby log evidence (repeated "failed to retrieve" warnings from frontends) should be correlated.

Minute 2022-03-20 15:58
- Top-8 components by anomaly events (count):
  1. checkoutservice-2 (43)
  2. checkoutservice-0 (29)
  3. checkoutservice-1 (28)
  4. frontend-2 (14)
  5. currencyservice-2 (14)
  6. adservice2 (13)
  7. cartservice-1 (11)
  8. cartservice-2 (11)

- Metric evidence (most severe per component, up to 5):
  - checkoutservice-0: notable CPU throttling metric (container_cpu_cfs_throttled_...) with a meaningful breach_ratio (~0.127) — suggests CPU throttling at 15:58.
  - checkoutservice-1 / checkoutservice-2: mesh-related metrics (mesh.source/destination) showing breaches.
  - currencyservice-2: container memory_failures.* flagged with non-trivial breach_ratio (memory-related issue).
  - Other checkoutservice entries show smaller positive deltas in mesh/CPU metrics.

- Trace evidence:
  - checkoutservice and frontend traces include duration/row_count anomalies (e.g., trace.to_currencyservice-2.duration, trace.from_frontend-1.duration) — indicating latency/throughput changes across service calls.
  - frontend-2 shows small row_count delta at this minute.

- Error logs at exact minute:
  - No error_logs entries exactly at 15:58 for these top components (error_log_count_in_minute = 0).

Interpretation for 15:58:
- This cluster is dominated by checkoutservice pods and shows CPU throttling and mesh-related anomalies, plus memory-failure signals on currencyservice — consistent with resource pressure or service-mesh degradation affecting checkoutservice and downstream services around 15:58.
- As with 15:49, direct error-log hits at the exact minute are absent, but earlier frontend logs (repeated "failed to retrieve") align temporally with the overall incident window and suggest broader request failures or backend unavailability.

Overall actionable conclusions and next steps
- Two focal minutes to investigate first: 15:49 (frontend-heavy traffic/trace spikes + mesh/CPU anomalies) and 15:58 (checkoutservice-heavy CPU throttling/mesh and memory-failure signals).
- Prioritize:
  1. Inspect full trace details and Istio/mesh logs around 15:49 to see which backend endpoints were hit and whether routing/latency spikes or retries occurred.
  2. Pull full metric time series (CPU throttling, cpu_usage, memory_failures, memory_rss, network I/O, mesh metrics) for the implicated pods and their host nodes from ~15:40–16:00 to confirm resource contention or node-level issues.
  3. Correlate frontend "failed to retrieve" log messages (seen widely across the incident window) with the trace destinations identified at 15:49 and 15:58 to determine if backend failures caused frontend errors or vice versa.
  4. Check Kubernetes events and pod lifecycle around these minutes for OOMKills, throttling warnings, or pod restarts.
- If you want, I can now extract the raw time-series and full logs for the top components at 15:49 and 15:58 for deeper forensic correlation.

The original code execution output of IPython Kernel is also provided below for reference:

(                   cmdb_id  event_count
0               frontend-1           49
1               frontend-0           26
2              frontend2-0           26
3       checkoutservice2-0           24
4  productcatalogservice-0           23
5  productcatalogservice-2           23
6        checkoutservice-0           19
7  productcatalogservice-1           15,            minute_timestamp                  cmdb_id  event_count                                   metric_anomalies                                    trace_anomalies  error_log_count_in_minute error_message_samples
0  2022-03-20 15:49:00+0800               frontend-1           49  [{'kpi_name': 'mesh.source.frontend.jaeger-col...  [{'trace_name': 'trace.to_currencyservice-0.ro...                          0                    []
1  2022-03-20 15:49:00+0800               frontend-0           26  [{'kpi_name': 'container.node-5.container_cpu_...  [{'trace_name': 'trace.to_checkoutservice-0.du...                          0                    []
2  2022-03-20 15:49:00+0800              frontend2-0           26  [{'kpi_name': 'container.node-6.container_cpu_...  [{'trace_name': 'trace.self.row_count', 'value...                          0                    []
3  2022-03-20 15:49:00+0800       checkoutservice2-0           24  [{'kpi_name': 'mesh.destination.frontend2.chec...  [{'trace_name': 'trace.from_root.duration_p95'...                          0                    []
4  2022-03-20 15:49:00+0800  productcatalogservice-0           23  [{'kpi_name': 'mesh.destination.recommendation...  [{'trace_name': 'trace.from_frontend-2.row_cou...                          0                    []
5  2022-03-20 15:49:00+0800  productcatalogservice-2           23  [{'kpi_name': 'mesh.destination.recommendation...  [{'trace_name': 'trace.from_frontend-2.row_cou...                          0                    []
6  2022-03-20 15:49:00+0800        checkoutservice-0           19  [{'kpi_name': 'mesh.destination.frontend.check...  [{'trace_name': 'trace.self.row_count', 'value...                          0                    []
7  2022-03-20 15:49:00+0800  productcatalogservice-1           15  [{'kpi_name': 'mesh.destination.recommendation...  [{'trace_name': 'trace.from_frontend-2.row_cou...                          0                    [],              cmdb_id  event_count
0  checkoutservice-2           43
1  checkoutservice-0           29
2  checkoutservice-1           28
3         frontend-2           14
4  currencyservice-2           14
5         adservice2           13
6      cartservice-1           11
7      cartservice-2           11,            minute_timestamp            cmdb_id  event_count                                   metric_anomalies                                    trace_anomalies  error_log_count_in_minute error_message_samples
0  2022-03-20 15:58:00+0800  checkoutservice-2           43  [{'kpi_name': 'mesh.source.checkoutservice.cur...  [{'trace_name': 'trace.to_currencyservice-2.du...                          0                    []
1  2022-03-20 15:58:00+0800  checkoutservice-0           29  [{'kpi_name': 'container.node-5.container_cpu_...  [{'trace_name': 'trace.from_frontend-1.duratio...                          0                    []
2  2022-03-20 15:58:00+0800  checkoutservice-1           28  [{'kpi_name': 'mesh.destination.frontend.check...  [{'trace_name': 'trace.to_emailservice-0.durat...                          0                    []
3  2022-03-20 15:58:00+0800         frontend-2           14  [{'kpi_name': 'container.node-5.container_netw...  [{'trace_name': 'trace.from_root.row_count', '...                          0                    []
4  2022-03-20 15:58:00+0800  currencyservice-2           14  [{'kpi_name': 'container.node-5.container_memo...  [{'trace_name': 'trace.from_checkoutservice-2....                          0                    []
5  2022-03-20 15:58:00+0800         adservice2           13  [{'kpi_name': 'runtime.java_lang_Threading_Cur...                                                 []                          0                    []
6  2022-03-20 15:58:00+0800      cartservice-1           11  [{'kpi_name': 'container.node-5.container_memo...                                                 []                          0                    []
7  2022-03-20 15:58:00+0800      cartservice-2           11  [{'kpi_name': 'mesh.destination.checkoutservic...                                                 []                          0                    [])```
```